# 🎲 04 · Noise, statistics and inference
### How to find a signal you cannot see: averaging, hypothesis tests, mixture models, the EM algorithm and clustering metrics

**BiologyToResearch** · Part A — Foundations · Module 04 of 34

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/cryoem/tutorial/BiologyToResearch/notebooks/04_noise_statistics_inference.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, SciPy (`scipy.stats`, `scipy.optimize`, `scipy.cluster`), scikit-learn, Matplotlib |
| **Prerequisites** | Modules 01–03 (arrays, rotations and rotation-invariant shells, Fourier basics). No statistics course needed. |
| **Estimated time** | 120 min (reading + running + the ✏️ exercises) |
| **Lab papers covered** | Statistical foundations of FAML (Zhao *et al.* 2018), Multi-Pattern Pursuit (Xu *et al.* 2019), DISCA (Zeng *et al.* 2023), the fungal plasma-membrane study (Jiang *et al.* 2025) and the classification scheme of Xu, Beck & Alber (2012); SNR definition of BOE-ViT (2025) — see [`PAPER_ATLAS.md`](../PAPER_ATLAS.md) |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis. All
> experiments in this course are small, controlled *toy versions* of published methods. They are designed to teach the
> core idea of each paper; they do **not** reproduce the papers' numbers. **Nothing is stored in the repository:**
> molecules are real Protein Data Bank (PDB) models, downloaded the first time a notebook needs them and cached on
> your computer (offline, code-built stand-ins of the same size and symmetry are used instead); all images and data
> are simulated by code. The "membrane" point patterns in Section 5 are synthetic numbers *inspired by* a published
> study, not its data.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Describe** noise as a random variable and compute its mean, variance and standard deviation.
2. **Compute** the signal-to-noise ratio (SNR) in two ways — as a variance ratio and from the correlation of two noisy
   copies, $\mathrm{SNR}=c/(1-c)$ — and **measure** how averaging $N$ aligned copies improves the SNR $N$-fold.
3. **Run and interpret** a permutation test, a Kruskal–Wallis test (with Dunn's post-hoc comparisons) and a Wilcoxon
   signed-rank test, and **explain** why the *unit of replication* matters.
4. **Implement** the expectation–maximization (EM) algorithm for Gaussian mixture models by hand (1D and 2D) and
   **verify** it against scikit-learn.
5. **Choose** the number of clusters with BIC, and **compare** GMM, k-means and hierarchical clustering with silhouette.
6. **Evaluate** a clustering with Hungarian-matched accuracy and the adjusted Rand index, and a detector with
   precision, recall and F1.
7. **Connect** each tool to the Xu-lab paper that uses it (FAML, Multi-Pattern Pursuit, DISCA, the fungal-membrane study).

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the molecular zoo | 5 min |
| 1 · Random variables and noise | histograms, mean, variance, the law of large numbers | 10 min |
| 2 · Similarity: Pearson correlation | the score behind template matching and alignment | 8 min |
| 3 · Signal-to-noise ratio, two ways | variance ratio; $c/(1-c)$ from two noisy copies | 10 min |
| 4 · Averaging: the √N law | why thousands of copies are needed — and why they must be aligned | 10 min |
| 5 · Hypothesis testing | permutation, Kruskal–Wallis + Dunn, Wilcoxon signed-rank | 25 min |
| 6 · Likelihood | the idea behind every "ML" method | 7 min |
| 7 · Mixture models and EM | code EM by hand (1D, then 2D) | 20 min |
| 8 · How many clusters? BIC | model selection as in DISCA | 7 min |
| 9 · k-means, hierarchical clustering, silhouette | the classical toolbox (Xu 2012) | 10 min |
| 10 · Scoring results | Hungarian accuracy, ARI, precision/recall/F1 | 10 min |
| ✅ / 🧠 / 🚀 | checkpoint, discussion, going further | 10 min |

## 🧫 The biological question

Imagine you have imaged a bacterium by cryo-electron tomography. Somewhere in that grey, grainy 3D picture there are
thousands of ribosomes, hundreds of proteasomes, and molecules nobody has named yet. In a single **subtomogram** (a small
cube cut around one particle) the signal is buried: the **signal-to-noise ratio is about 0.01–0.1**, meaning the random
fluctuations are 3 to 10 times *larger* than the molecule's own contrast. No human — and no algorithm — can read the
shape of a molecule from one such cube.

And yet the Xu Lab and others recover 1–3 nm structures from such data, sort particles into species without labels, and
decide whether a drug changes how proteins are arranged in a membrane. They do it with **statistics**: averaging many
copies, testing whether differences are real, fitting probabilistic models with hidden variables, and scoring the
results honestly. This notebook teaches exactly those tools, with the molecules of our zoo.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, this notebook only needs a CPU.")
try:
    import torch
    print("PyTorch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch not installed (not needed in this notebook).")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("scipy", "scipy"), ("matplotlib", "matplotlib"),
                         ("sklearn", "scikit-learn")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
import os
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Ellipse
from scipy import ndimage, stats
from scipy.cluster import hierarchy
from scipy.spatial import cKDTree
from scipy.spatial.distance import squareform
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.mixture import GaussianMixture

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "image.cmap": "gray"})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py): building
zoo maps, rotating volumes and images, adding noise at a chosen SNR, Pearson correlation, and the Hungarian-matched
clustering accuracy we derive in Section 10. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v3.0; copied so this notebook runs alone) ──
import os
import struct

import numpy as np
from scipy import ndimage

ZOO = {
    "4v4r": dict(name="70S ribosome", organism="bacterial (E. coli / T. thermophilus)", mass_kDa=2330, symmetry="C1"),
    "4v7r": dict(name="80S ribosome", organism="S. cerevisiae", mass_kDa=3330, symmetry="C1"),
    "5mrc": dict(name="mitochondrial ribosome", organism="S. cerevisiae", mass_kDa=3320, symmetry="C1"),
    "3j9i": dict(name="20S proteasome", organism="Thermoplasma acidophilum", mass_kDa=710, symmetry="D7"),
    "6utj": dict(name="singly capped proteasome", organism="Thermoplasma acidophilum", mass_kDa=910, symmetry="C7"),
    "4cr2": dict(name="26S proteasome", organism="S. cerevisiae", mass_kDa=1280, symmetry="C1"),
    "4v94": dict(name="TRiC/CCT chaperonin (×2)", organism="S. cerevisiae", mass_kDa=1910, symmetry="C2"),
    "1bxn": dict(name="RuBisCO (L4S4 half)", organism="Cupriavidus necator", mass_kDa=290, symmetry="C4"),
    "1qvr": dict(name="ClpB (3 of 6 subunits)", organism="Thermus thermophilus", mass_kDa=290, symmetry="C1"),
    "3cf3": dict(name="p97/VCP ATPase", organism="Mus musculus", mass_kDa=570, symmetry="C6"),
    "2cg9": dict(name="Hsp90–Sba1 chaperone", organism="S. cerevisiae", mass_kDa=190, symmetry="C2"),
    "1u6g": dict(name="Cand1–Cul1–Roc1 ligase", organism="H. sapiens", mass_kDa=260, symmetry="C1"),
    "3d2f": dict(name="Sse1p–Hsp70 chaperone", organism="S. cerevisiae / H. sapiens", mass_kDa=230, symmetry="C1"),
    "1s3x": dict(name="Hsp70 ATPase domain", organism="H. sapiens", mass_kDa=44, symmetry="C1"),
}
_EXTRA_ENTRIES = {"5a1a": dict(name="β-galactosidase", organism="E. coli", mass_kDa=465, symmetry="D2")}
_BOND = dict(N_CA=1.458, CA_C=1.525, C_N=1.329, C_O=1.231)
_ANGLE = dict(N_CA_C=111.2, CA_C_N=116.2, C_N_CA=121.7, CA_C_O=120.5)
_SIDECHAIN = {
    "GLY": [], "ALA": [], "SER": [("OG", "O")], "CYS": [("SG", "S")], "VAL": [("CG1", "C"), ("CG2", "C")],
    "THR": [("OG1", "O"), ("CG2", "C")], "PRO": [("CG", "C"), ("CD", "C")],
    "LEU": [("CG", "C"), ("CD1", "C"), ("CD2", "C")], "ILE": [("CG1", "C"), ("CG2", "C"), ("CD1", "C")],
    "ASP": [("CG", "C"), ("OD1", "O"), ("OD2", "O")], "ASN": [("CG", "C"), ("OD1", "O"), ("ND2", "N")],
    "GLU": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("OE2", "O")], "GLN": [("CG", "C"), ("CD", "C"), ("OE1", "O"), ("NE2", "N")],
    "LYS": [("CG", "C"), ("CD", "C"), ("CE", "C"), ("NZ", "N")], "MET": [("CG", "C"), ("SD", "S"), ("CE", "C")],
    "ARG": [("CG", "C"), ("CD", "C"), ("NE", "N"), ("CZ", "C"), ("NH1", "N"), ("NH2", "N")],
    "HIS": [("CG", "C"), ("ND1", "N"), ("CD2", "C"), ("CE1", "C"), ("NE2", "N")],
    "PHE": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C")],
    "TYR": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("CE1", "C"), ("CE2", "C"), ("CZ", "C"), ("OH", "O")],
    "TRP": [("CG", "C"), ("CD1", "C"), ("CD2", "C"), ("NE1", "N"), ("CE2", "C"), ("CE3", "C"), ("CZ2", "C"),
            ("CZ3", "C"), ("CH2", "C")],
}
_AA_FREQ = {"LEU": 9.9, "ALA": 8.3, "GLY": 7.1, "VAL": 6.9, "GLU": 6.8, "SER": 6.6, "ILE": 5.9, "LYS": 5.8, "ARG": 5.5,
            "ASP": 5.5, "THR": 5.3, "PRO": 4.7, "ASN": 4.1, "GLN": 3.9, "PHE": 3.9, "TYR": 2.9, "MET": 2.4, "HIS": 2.3,
            "CYS": 1.4, "TRP": 1.1}
def _nerf(a, b, c, bond, angle, torsion):
    """Place atom d so that |cd| = bond, angle(b,c,d) = angle and dihedral(a,b,c,d) = torsion (degrees)."""
    bc = (c - b) / np.linalg.norm(c - b)
    n = np.cross(b - a, bc)
    n /= np.linalg.norm(n)
    m = np.cross(n, bc)
    ang, tor = np.radians(angle), np.radians(torsion)
    d = np.array([-bond * np.cos(ang), bond * np.sin(ang) * np.cos(tor), bond * np.sin(ang) * np.sin(tor)])
    return c + d[0] * bc + d[1] * m + d[2] * n


def build_backbone(phi, psi, omega=None) -> np.ndarray:
    """Backbone atoms N, CA, C, O, CB (array n×5×3, Å) of a chain with the given torsion angles (degrees)."""
    n = len(phi)
    omega = np.full(n, 180.0) if omega is None else np.asarray(omega, float)
    N = [np.zeros(3)]
    CA = [np.array([_BOND["N_CA"], 0.0, 0.0])]
    C = [_nerf(np.array([0.0, 1.0, 0.0]), N[0], CA[0], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[0])]
    for i in range(n - 1):
        N.append(_nerf(N[i], CA[i], C[i], _BOND["C_N"], _ANGLE["CA_C_N"], psi[i]))
        CA.append(_nerf(CA[i], C[i], N[i + 1], _BOND["N_CA"], _ANGLE["C_N_CA"], omega[i]))
        C.append(_nerf(C[i], N[i + 1], CA[i + 1], _BOND["CA_C"], _ANGLE["N_CA_C"], phi[i + 1]))
    N, CA, C = np.array(N), np.array(CA), np.array(C)
    O = np.array([_nerf(N[i], CA[i], C[i], _BOND["C_O"], _ANGLE["CA_C_O"], psi[i] + 180.0) for i in range(n)])
    b, c = CA - N, C - CA
    a = np.cross(b, c)
    CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA      # ideal Cβ position
    return np.stack([N, CA, C, O, CB], 1)


def _ideal_helix_template(n=160):
    """An ideal α-helix (φ = −57.8°, ψ = −47.0°) with its axis along +z through x = y = 0."""
    bb = build_backbone(np.full(n, -57.8), np.full(n, -47.0))
    ca = bb[:, 1]
    centre = ca.mean(0)
    u = np.linalg.svd(ca - centre)[2][0]
    if np.dot(ca[-1] - ca[0], u) < 0:
        u = -u
    # rotation taking u to z
    v = np.cross(u, [0, 0, 1.0]); s = np.linalg.norm(v); cth = u[2]
    K = np.array([[0, -v[2], v[1]], [v[2], 0, -v[0]], [-v[1], v[0], 0]])
    Rm = np.eye(3) + K + K @ K * ((1 - cth) / (s ** 2 + 1e-12))
    xyz = (bb - centre) @ Rm.T
    axis_xy = xyz[:, 1, :2].mean(0)
    xyz[..., :2] -= axis_xy
    return xyz


_HELIX = _ideal_helix_template()
def _sidechains(bb, resnames, rng):
    """Approximate side-chain heavy atoms grown outward from Cβ (positions are schematic; counts/elements are right)."""
    names, elems, xyz, resid = [], [], [], []
    ca, cb = bb[:, 1], bb[:, 4]
    out = cb - ca
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    for i, rn in enumerate(resnames):
        sc = _SIDECHAIN[rn]
        if not sc:
            continue
        perp = np.cross(out[i], rng.normal(size=3)); perp /= np.linalg.norm(perp)
        for k, (an, el) in enumerate(sc):
            depth = 1.0 + (k // 2)                               # schematic zig-zag outward
            pos = cb[i] + 1.5 * depth * out[i] + 0.9 * (1 if k % 2 else -1) * perp
            names.append(an); elems.append(el); xyz.append(pos); resid.append(i)
    return names, elems, np.array(xyz).reshape(-1, 3), np.array(resid, int)


def _loop_path(a, b, outward, step=3.8):
    """Cα positions of a loop from Cα `a` to Cα `b`: a circular arc bulging along `outward`, cut into equal chords so
    that every Cα–Cα distance (including the two ends) is exactly `step` Å."""
    c = np.linalg.norm(b - a)
    e = (b - a) / c
    u = outward - (outward @ e) * e                                  # bulge direction ⟂ to the chord
    if np.linalg.norm(u) < 1e-6:
        u = np.cross(e, [0.0, 0.0, 1.0]) if abs(e[2]) < 0.9 else np.cross(e, [1.0, 0.0, 0.0])
    u = u / np.linalg.norm(u)
    m = max(int(np.ceil(c / step)), 1)                               # m new residues → n = m + 1 equal chords
    n = m + 1
    lo, hi = 1e-9, 2 * np.pi - 1e-9                                  # arc angle Θ: sin(Θ/2) / sin(Θ/2n) = c / step
    for _ in range(60):
        th = (lo + hi) / 2
        lo, hi = (th, hi) if np.sin(th / 2) / np.sin(th / (2 * n)) > c / step else (lo, th)
    R = step / (2 * np.sin(th / (2 * n)))
    centre = (a + b) / 2 - R * np.cos(th / 2) * u
    beta = np.pi / 2 + th / 2 - th * np.arange(1, n) / n             # angles of the new points, a → b over the bulge
    return centre + R * (np.outer(np.cos(beta), e) + np.outer(np.sin(beta), u))


def _assemble_chain(ca_path_segments, rng, chain="A", start_resnum=1):
    """Turn helices (n×5×3 arrays) joined by loops into one chain with residue names and side chains."""
    blocks = []
    for k, seg in enumerate(ca_path_segments):
        blocks.append(seg)
        if k + 1 < len(ca_path_segments):
            a, b = seg[-1, 1], ca_path_segments[k + 1][0, 1]
            path = _loop_path(a, b, (a + b) / 2 - seg.reshape(-1, 3).mean(0))
            m = len(path)
            tang = np.gradient(path, axis=0) if m > 1 else np.array([b - a])
            tang /= np.linalg.norm(tang, axis=1, keepdims=True)
            side = np.cross(tang, rng.normal(size=3)); side /= np.linalg.norm(side, axis=1, keepdims=True)
            loop = np.stack([path - 1.2 * tang, path, path + 1.2 * tang, path + 1.2 * tang + 1.0 * side,
                             path - 1.3 * side], 1)
            blocks.append(loop)
    bb = np.concatenate(blocks, 0)
    aa = list(_AA_FREQ)
    p = np.array([_AA_FREQ[a] for a in aa]); p /= p.sum()
    resnames = rng.choice(aa, len(bb), p=p)
    bb_names = ["N", "CA", "C", "O", "CB"]
    names, elems, xyz, resid = [], [], [], []
    for i in range(len(bb)):
        for j, an in enumerate(bb_names):
            if an == "CB" and resnames[i] == "GLY":
                continue
            names.append(an); elems.append(an[0]); xyz.append(bb[i, j]); resid.append(i)
    sn, se, sx, sr = _sidechains(bb, resnames, rng)
    names += sn; elems += se; resid = np.concatenate([np.array(resid, int), sr])
    xyz = np.concatenate([np.array(xyz), sx], 0)
    order = np.argsort(resid, kind="stable")
    return dict(xyz=xyz[order].astype(np.float32), name=np.array(names)[order], element=np.array(elems)[order],
                resname=resnames[resid[order]], resnum=(resid[order] + start_resnum).astype(int),
                chain=np.full(len(order), chain))


def helix_bundle(radii_A, rng, chain="A", spacing=10.0):
    """A globular 'domain' that fills an ellipsoid (semi-axes in Å, long axis = local z) with antiparallel helices."""
    a, b, c = radii_A[0] + 2.5, radii_A[1] + 2.5, radii_A[2] + 1.5   # helices stop short of the surface: pad
    rows = []
    j = 0
    while (j * spacing * np.sqrt(3) / 2) <= b:
        rows.append(j); rows.append(-j) if j else None; j += 1
    pts = []
    for jj in sorted(set(rows)):
        y = jj * spacing * np.sqrt(3) / 2
        xs = np.arange(-a, a + 1e-6, spacing) + (spacing / 2 if jj % 2 else 0)
        row = [(x, y) for x in xs if (x / a) ** 2 + (y / b) ** 2 < 1]
        pts.append(row if (jj % 2 == 0) else row[::-1])
    pts = [p for row in pts for p in row]
    segs = []
    for k, (x, y) in enumerate(pts):
        h = c * np.sqrt(max(0.0, 1 - (x / a) ** 2 - (y / b) ** 2)) - 2.0
        n = int(2 * h / 1.5)
        if n < 6:
            continue
        seg = _HELIX[:n].copy()
        seg[..., 2] -= seg[:, 1, 2].mean()
        phase = rng.uniform(0, 2 * np.pi)
        Rz = np.array([[np.cos(phase), -np.sin(phase), 0], [np.sin(phase), np.cos(phase), 0], [0, 0, 1]])
        seg = seg @ Rz.T
        if len(segs) % 2 == 1:                                   # antiparallel neighbours: 180° turn about x
            seg = seg * np.array([1.0, -1.0, -1.0])
        seg = seg + np.array([x, y, rng.uniform(-1.0, 1.0)])
        segs.append(seg)
    return _assemble_chain(segs, rng, chain)


def _transform(atoms, R=np.eye(3), t=(0, 0, 0)):
    out = dict(atoms)
    out["xyz"] = (atoms["xyz"] @ np.asarray(R, float).T + np.asarray(t, float)).astype(np.float32)
    return out


def merge_atoms(parts):
    keys = parts[0].keys()
    return {k: np.concatenate([p[k] for p in parts]) for k in keys}


def _rz(a):
    return np.array([[np.cos(a), -np.sin(a), 0], [np.sin(a), np.cos(a), 0], [0, 0, 1.0]])


def _rx(a):
    return np.array([[1, 0, 0], [0, np.cos(a), -np.sin(a)], [0, np.sin(a), np.cos(a)]])


def _ry(a):
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


_CHAIN_IDS = "ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789"
def _ring(n, radius, z, radii, phase=0.0, tilt=None):
    """n identical subunits (semi-axes `radii` nm: radial, tangential, axial) around the z axis."""
    out = []
    for k in range(n):
        a = phase + 2 * np.pi * k / n
        R = _rz(a) if tilt is None else _rz(a) @ tilt
        out.append(dict(radii=radii, centre=(radius * np.cos(a), radius * np.sin(a), z), R=R, key=("ring", radii, tilt is None)))
    return out


def _lobe(radii, centre, R=None):
    return [dict(radii=radii, centre=centre, R=np.eye(3) if R is None else R, key=None)]


def _dihedral(subunits):
    """Add the partners generated by a 2-fold axis along x (dihedral symmetry)."""
    Rx = _rx(np.pi)
    return subunits + [dict(radii=s["radii"], centre=tuple(Rx @ np.array(s["centre"])), R=Rx @ s["R"], key=s["key"])
                       for s in subunits]


def _blueprint(pid):
    """Subunit layout (nm) of the code-generated stand-in for each zoo entry."""
    if pid in ("4v4r", "4v7r", "5mrc"):
        s = {"4v4r": 1.0, "4v7r": 1.13, "5mrc": 1.14}[pid]
        L = [_lobe((9.0 * s, 8.0 * s, 6.5 * s), (0, 0, 4.5 * s)),                       # large subunit body
             _lobe((3.0 * s, 3.0 * s, 3.0 * s), (0, 1.0 * s, 11.0 * s)),                # central protuberance
             _lobe((2.0 * s, 2.0 * s, 3.5 * s), (-9.0 * s, 0, 7.0 * s), _ry(-0.6)),     # L1-like stalk
             _lobe((1.4 * s, 1.4 * s, 4.0 * s), (9.5 * s, 0, 6.5 * s), _ry(0.7)),       # L7/L12-like stalk
             _lobe((7.5 * s, 5.0 * s, 3.8 * s), (1.0 * s, 0, -4.5 * s)),                # small subunit body
             _lobe((3.5 * s, 3.2 * s, 3.0 * s), (-5.5 * s, -1.0 * s, -6.5 * s))]        # small subunit head
        if pid == "4v7r":                                                                # eukaryotic expansion segments
            L += [_lobe((3.2, 2.6, 2.8), (8.0, 7.0, 7.0)), _lobe((3.0, 2.6, 3.2), (-6.0, -8.0, -6.5)),
                  _lobe((2.4, 2.4, 3.6), (-10.5, 4.0, -1.0), _ry(0.5))]
        if pid == "5mrc":                                                                # mitochondria-specific proteins
            L += [_lobe((4.2, 3.6, 2.8), (0, -7.5, 12.5)), _lobe((3.4, 3.0, 3.4), (9.0, -6.0, -8.0)),
                  _lobe((3.0, 3.0, 3.0), (-9.5, 7.0, 1.0)), _lobe((2.5, 4.5, 2.0), (2.0, 9.0, 9.0))]
        prng = np.random.default_rng(_seed(pid) + 1)                                     # rRNA-like surface bumps
        for _ in range(8):
            d = prng.normal(size=3); d /= np.linalg.norm(d)
            L += [_lobe(tuple(prng.uniform(1.2, 1.9, 3)), tuple(s * d * np.array([9.5, 8.0, 10.5])), _rz(prng.uniform(0, 6)))]
        return [x for l in L for x in l]
    if pid in ("3j9i", "6utj", "4cr2"):
        core = []
        for z, radii, ph in [(1.9, (1.9, 1.7, 2.0), 0.0), (5.6, (2.1, 1.7, 1.85), np.pi / 7)]:
            core += _ring(7, 3.6, z, radii, ph)
        core = _dihedral(core)                                                           # α7β7β7α7, D7
        if pid == "6utj":
            core += _ring(7, 3.3, 10.0, (1.9, 1.6, 2.4), np.pi / 14)                     # one heptameric cap: C7
        if pid == "4cr2":
            core += _ring(6, 4.3, 9.6, (2.2, 1.8, 2.0), 0.3)                             # base ring of six
            core += (_lobe((4.0, 3.0, 3.0), (3.5, 0, 14.5)) + _lobe((3.0, 2.5, 3.5), (-1.5, 3.0, 15.5))
                     + _lobe((2.5, 2.5, 2.5), (-3.5, -2.0, 13.5)) + _lobe((3.0, 3.0, 2.5), (-4.5, 0.5, 10.5)))
        return core
    if pid == "4v94":
        one = []
        for z in (3.6, -3.6):
            one += _ring(8, 5.2, z, (2.6, 2.0, 3.4), np.pi / 8 if z < 0 else 0.0)
        two = []
        for dx, rot in [(-8.5, 0.0), (8.5, np.pi)]:
            for s in one:
                c = _rz(rot) @ np.array(s["centre"])
                two.append(dict(radii=s["radii"], centre=(c[0] + dx, c[1], c[2]), R=_rz(rot) @ s["R"], key=s["key"]))
        return two
    if pid == "1bxn":
        # like the deposited file (the crystal's asymmetric unit): one L4S4 layer = half of the L8S8 enzyme, C4.
        # `_dihedral` of this list would give the complete enzyme (D4).
        return _ring(4, 3.6, 2.2, (2.6, 2.4, 2.4)) + _ring(4, 2.6, 5.0, (1.9, 1.7, 1.35), np.pi / 4)
    if pid == "1qvr":
        # like the deposited file: three neighbouring subunits (half of the C6 hexameric ring), C1
        tiers = [_ring(6, 4.2, 2.2, (2.2, 2.0, 2.0)), _ring(6, 3.9, -2.2, (2.3, 2.0, 2.0), np.pi / 6),
                 _ring(6, 7.2, 2.4, (0.9, 0.9, 4.6), 0.4, tilt=_rx(1.2)),               # coiled-coil 'propellers'
                 _ring(6, 3.8, 6.6, (1.7, 1.6, 1.8), 0.5)]                               # N-domains on top
        return [s for t in tiers for s in t[:3]]
    if pid == "3cf3":
        return (_ring(6, 4.0, 2.0, (2.1, 2.0, 2.0)) + _ring(6, 4.0, -2.2, (2.2, 2.0, 2.1), np.pi / 6)
                + _ring(6, 6.6, 3.0, (2.0, 1.6, 1.8), 0.25))                             # N-domains flush: C6
    if pid == "2cg9":
        half = (_lobe((1.9, 1.7, 1.7), (2.2, 0.8, 6.0)) + _lobe((2.2, 1.8, 2.0), (2.6, -0.4, 2.0))
                + _lobe((1.7, 1.5, 1.8), (1.4, 0.3, -2.2)) + _lobe((1.4, 1.2, 1.2), (4.2, 1.6, 7.6)))
        Rz = _rz(np.pi)
        return half + [dict(radii=h["radii"], centre=tuple(Rz @ np.array(h["centre"])), R=Rz @ h["R"], key=None) for h in half]
    if pid == "1u6g":
        out = []
        for k in range(6):                                                                # curved cullin stalk
            t = k / 5
            out += _lobe((1.4, 1.4, 1.6), (8.0 * np.sin(0.9 * t) - 3.0, 0, 9.0 * t - 4.5), _ry(0.9 * t))
        out += _lobe((2.2, 2.0, 2.0), (4.0, 0, 5.8)) + _lobe((1.2, 1.1, 1.1), (6.0, 1.8, 6.5))
        for k in range(10):                                                               # Cand1-like superhelix
            a = 1.6 * np.pi * k / 9
            out += _lobe((1.5, 1.4, 1.4), (-3.0 + 7.0 * np.sin(0.9 * k / 9) + 2.6 * np.cos(a), 2.6 * np.sin(a), 9.0 * k / 9 - 4.5))
        return out
    if pid == "3d2f":
        comp = (_lobe((1.8, 1.6, 1.8), (0, 0, 0)) + _lobe((1.8, 1.6, 1.8), (0, 3.2, 0)) + _lobe((1.6, 1.4, 2.4), (3.0, 1.6, 1.8))
                + _lobe((1.6, 1.5, 1.6), (-3.2, 1.0, -1.0)) + _lobe((1.6, 1.5, 1.6), (-3.2, 3.6, -0.6)))
        R2, t2 = _rx(2.3) @ _rz(0.7), np.array([1.5, -4.5, 5.5])
        return comp + [dict(radii=c["radii"], centre=tuple(R2 @ np.array(c["centre"]) + t2), R=R2 @ c["R"], key=None) for c in comp]
    if pid == "1s3x":
        return (_lobe((1.4, 1.4, 1.6), (-1.6, 0, -1.2)) + _lobe((1.3, 1.3, 1.5), (-1.4, 0.2, 1.6))
                + _lobe((1.4, 1.4, 1.6), (1.6, 0, -1.2)) + _lobe((1.2, 1.3, 1.4), (1.5, -0.3, 1.8)))
    raise KeyError(pid)


def _seed(pid):
    return int.from_bytes(pid.encode(), "little") % (2 ** 31)


_SCALE = {"4v4r": 0.923, "4v7r": 0.906, "5mrc": 0.883, "3j9i": 0.930, "6utj": 0.930, "4cr2": 0.930, "4v94": 0.886,
          "1bxn": 0.930, "1qvr": 0.885, "3cf3": 0.935, "2cg9": 0.995, "1u6g": 0.934, "3d2f": 0.997, "1s3x": 0.884}
def standin_atoms(pid):
    """Atomic model (dict of arrays) of the code-built stand-in for zoo entry `pid` (Å, centred at the origin).

    Used when the real entry cannot be downloaded: ideal α-helices arranged with the size, mass and symmetry of the
    PDB entry. Deterministic (the random numbers are seeded by the entry's name)."""
    pid = pid.lower()
    rng = np.random.default_rng(_seed(pid))
    built, parts = {}, []
    f = _SCALE.get(pid, 1.0)
    for k, s in enumerate(_blueprint(pid)):
        radii_A = tuple(10.0 * f * r for r in s["radii"])
        key = s["key"]
        if key is not None and key in built:
            sub = built[key]
        else:
            sub = helix_bundle(radii_A, rng)
            if key is not None:
                built[key] = sub
        part = _transform(sub, s["R"], 10.0 * f * np.array(s["centre"]))
        part["chain"] = np.full(len(part["xyz"]), _CHAIN_IDS[k % len(_CHAIN_IDS)])
        parts.append(part)
    atoms = merge_atoms(parts)
    if pid in ("3j9i", "6utj", "4cr2"):                             # lay the proteasome barrels along x (side view down z)
        atoms["xyz"] = (atoms["xyz"] @ _ry(np.pi / 2).T).astype(np.float32)
    atoms["xyz"] -= atoms["xyz"].mean(0)
    return atoms


def pdb_cache_dir():
    """Folder (outside the repository) where downloaded PDB files are kept."""
    return os.environ.get("BIO2RESEARCH_PDB_CACHE") or os.path.join(os.path.expanduser("~"), ".cache",
                                                                    "biology_to_research", "pdb")


def _pdb_urls(pid, kind):
    """Download addresses of one PDB file: kind = 'entry' or 'assembly1' (mmCIF) or 'pdb' (legacy PDB format)."""
    pid = pid.lower()
    mid = pid[1:3]
    if kind == "entry":
        return [f"https://files.rcsb.org/download/{pid.upper()}.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/mmCIF/{mid}/{pid}.cif.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/{pid}.cif"]
    if kind == "assembly1":
        return [f"https://files.rcsb.org/download/{pid.upper()}-assembly1.cif.gz",
                f"https://files.wwpdb.org/pub/pdb/data/assemblies/mmCIF/divided/{mid}/{pid}-assembly1.cif.gz"]
    if kind == "pdb":
        return [f"https://files.rcsb.org/download/{pid.upper()}.pdb.gz",
                f"https://files.wwpdb.org/pub/pdb/data/structures/divided/pdb/{mid}/pdb{pid}.ent.gz",
                f"https://www.ebi.ac.uk/pdbe/entry-files/download/pdb{pid}.ent"]
    raise ValueError(kind)


def fetch_pdb(pid, kind="entry", timeout=60):
    """Text of a PDB file, downloaded once and cached outside the repository. Returns (text, where_from), or
    (None, reason) when no mirror can be reached (or BIO2RESEARCH_STANDINS is set)."""
    import gzip
    import urllib.request
    name = {"entry": f"{pid.lower()}.cif.gz", "assembly1": f"{pid.lower()}-assembly1.cif.gz",
            "pdb": f"{pid.lower()}.pdb.gz"}[kind]
    path = os.path.join(pdb_cache_dir(), name)
    if os.path.exists(path):
        return gzip.decompress(open(path, "rb").read()).decode("utf-8", "replace"), f"cached copy ({path})"
    if os.environ.get("BIO2RESEARCH_STANDINS"):
        return None, "BIO2RESEARCH_STANDINS is set"
    reason = "no mirror tried"
    for url in _pdb_urls(pid, kind):
        try:
            with urllib.request.urlopen(url, timeout=timeout) as response:
                raw = response.read()
            text = (gzip.decompress(raw) if raw[:2] == b"\x1f\x8b" else raw).decode("utf-8", "replace")
            if ("_atom_site." not in text) and ("\nATOM  " not in text):
                raise ValueError("no atoms in the downloaded file")
            os.makedirs(pdb_cache_dir(), exist_ok=True)
            with open(path, "wb") as f:
                f.write(gzip.compress(text.encode()))
            return text, url
        except Exception as e:  # noqa: BLE001 — any failure: try the next mirror
            reason = f"{type(e).__name__}: {str(e)[:120]}"
    return None, reason


def _cif_tokens(line):
    """Split one line of an mmCIF loop into values (quoted values may contain spaces)."""
    if "'" not in line and '"' not in line:
        return line.split()
    import re
    toks = re.findall(r"'(?:[^']|'(?=\S))*'|\"(?:[^\"]|\"(?=\S))*\"|\S+", line)
    return [t[1:-1] if len(t) > 1 and t[0] == t[-1] and t[0] in "'\"" else t for t in toks]


_WATER = {"HOH", "DOD", "WAT", "H2O"}
def read_mmcif_atoms(text):
    """Atoms of the first model in an mmCIF file → dict(xyz, name, element, resname, chain, resnum).

    Reads the `_atom_site` loop; skips water and hydrogen, and keeps only the first of alternative atom locations."""
    lines = text.splitlines()
    i = next(k for k, line in enumerate(lines) if line.startswith("_atom_site."))
    cols = []
    while i < len(lines) and lines[i].startswith("_atom_site."):
        cols.append(lines[i].split()[0][len("_atom_site."):])
        i += 1
    c = {name: k for k, name in enumerate(cols)}
    get = lambda *names: next((c[n] for n in names if n in c), None)   # noqa: E731
    ix, iy, iz = c["Cartn_x"], c["Cartn_y"], c["Cartn_z"]
    i_el, i_name = get("type_symbol"), get("auth_atom_id", "label_atom_id")
    i_res, i_ch = get("auth_comp_id", "label_comp_id"), get("auth_asym_id", "label_asym_id")
    i_num, i_alt, i_mod = get("auth_seq_id", "label_seq_id"), get("label_alt_id"), get("pdbx_PDB_model_num")
    xyz, name, element, resname, chain, resnum = [], [], [], [], [], []
    first_model, buf = None, []
    for line in lines[i:]:
        if not line or line.startswith(("#", "_", "loop_", "data_")):
            break
        buf += _cif_tokens(line)
        while len(buf) >= len(cols):
            row, buf = buf[:len(cols)], buf[len(cols):]
            if i_mod is not None:
                first_model = row[i_mod] if first_model is None else first_model
                if row[i_mod] != first_model:
                    continue
            el = row[i_el].upper() if i_el is not None else row[i_name][:1].upper()
            if el in ("H", "D") or row[i_res] in _WATER:
                continue
            if i_alt is not None and row[i_alt] not in (".", "?", "A", "1"):
                continue
            xyz.append((float(row[ix]), float(row[iy]), float(row[iz])))
            name.append(row[i_name])
            element.append(el.capitalize() if len(el) > 1 else el)
            resname.append(row[i_res])
            chain.append(row[i_ch])
            num = row[i_num] if i_num is not None else "0"
            resnum.append(int(num) if num.lstrip("-").isdigit() else 0)
    return dict(xyz=np.array(xyz, np.float32), name=np.array(name), element=np.array(element),
                resname=np.array(resname), chain=np.array(chain), resnum=np.array(resnum))


_ATOMIC_MASS = {"H": 1.008, "C": 12.011, "N": 14.007, "O": 15.999, "P": 30.974, "S": 32.06, "Se": 78.97, "Mg": 24.305,
                "Zn": 65.38, "Fe": 55.845, "Mn": 54.938, "K": 39.098, "Na": 22.99, "Cl": 35.45, "Ca": 40.078}
def model_mass_kDa(atoms):
    """Mass of an atomic model without hydrogens, plus ~8 % for the missing hydrogens (kDa)."""
    return 1.08 * sum(_ATOMIC_MASS.get(e, 12.0) for e in atoms["element"]) / 1000.0


def canonical_frame(atoms):
    """Centre a model and rotate it onto its principal axes — longest extent along x, shortest along z — with the
    signs fixed by the skewness of the atom distribution. A proteasome barrel then lies along x and a ring
    (RuBisCO, p97) lies in the x–y plane, like the stand-ins."""
    xyz = atoms["xyz"].astype(np.float64)
    X = xyz - xyz.mean(0)
    evecs = np.linalg.eigh(np.cov(X.T))[1][:, ::-1]                 # columns: principal axes, largest first
    Y = X @ evecs
    sign = np.sign(np.sum(Y ** 3, axis=0))
    sign[sign == 0] = 1.0
    Y *= sign
    if np.linalg.det(evecs * sign) < 0:                             # keep a proper rotation (no mirror image)
        Y[:, 2] *= -1
    out = dict(atoms)
    out["xyz"] = Y.astype(np.float32)
    return out


def pdb_atoms(pid, verbose=False):
    """Real atomic model of a PDB entry, in the canonical frame. Returns (atoms, description), or (None, reason).

    Uses the deposited entry unless its mass is far from ZOO's reference mass, in which case the first biological
    assembly is tried and the closer of the two kept (e.g. one ribosome rather than the two of a crystal)."""
    pid = pid.lower()
    ref = ZOO.get(pid, {}).get("mass_kDa") or _EXTRA_ENTRIES.get(pid, {}).get("mass_kDa")
    candidates, reason = [], ""
    for kind in ("entry", "assembly1"):
        if candidates and (ref is None or abs(np.log(model_mass_kDa(candidates[0][0]) / ref)) < np.log(1.3)):
            break
        text, where = fetch_pdb(pid, kind)
        if text is None:
            reason = where
            continue
        try:
            atoms = read_mmcif_atoms(text)
        except (StopIteration, KeyError, ValueError) as e:
            reason = f"could not read the file ({type(e).__name__})"
            continue
        if len(atoms["xyz"]):
            label = "deposited entry" if kind == "entry" else "biological assembly 1"
            candidates.append((atoms, f"PDB {pid.upper()} ({label}; {len(atoms['xyz']):,} atoms; from {where})"))
    if not candidates:
        return None, reason
    atoms, desc = min(candidates, key=lambda a: abs(np.log(model_mass_kDa(a[0]) / ref)) if ref else 0.0)
    if verbose:
        print(desc)
    return canonical_frame(atoms), desc


_ATOMS_CACHE = {}
def zoo_atoms(pid):
    """Atomic model (dict of arrays, Å, centred) of zoo entry `pid`: the real PDB model when it can be downloaded, the
    code-built stand-in otherwise. The source is recorded in `atoms_source(pid)`."""
    pid = pid.lower()
    if pid not in _ATOMS_CACHE:
        atoms, desc = pdb_atoms(pid)
        if atoms is None:
            atoms, desc = standin_atoms(pid), f"code-built stand-in modelled on PDB {pid.upper()} (download failed: {desc})"
        _ATOMS_CACHE[pid] = (atoms, desc)
    return _ATOMS_CACHE[pid][0]


def atoms_source(pid):
    """Where the atoms of zoo entry `pid` came from (a real PDB file or the offline stand-in)."""
    zoo_atoms(pid)
    return _ATOMS_CACHE[pid.lower()][1]


_Z = {"C": 6.0, "N": 7.0, "O": 8.0, "S": 16.0, "P": 15.0, "H": 1.0, "Mg": 12.0, "Zn": 30.0, "Fe": 26.0, "K": 19.0,
      "Na": 11.0, "Cl": 17.0, "Ca": 20.0, "Mn": 25.0, "Se": 34.0}
def atoms_to_map(xyz, weights=None, voxel=10.0, box=None, sigma_A=None):
    """Density map [z, y, x] from atom positions (Å): trilinear splatting on a grid twice as fine, Gaussian blur
    (σ = `sigma_A`, default = voxel/2), then 2×2×2 binning to `voxel` Å. Centred on the box if `box` is given."""
    xyz = np.asarray(xyz, float)
    w = np.ones(len(xyz)) if weights is None else np.asarray(weights, float)
    fine = voxel / 2.0
    if box is None:
        ext = np.ptp(xyz, 0).max() + 6 * voxel
        box = int(np.ceil(ext / voxel)) + 1
    n = 2 * box
    centre = (n - 1) / 2.0
    p = xyz[:, ::-1] / fine + centre                                  # (z, y, x) grid coordinates
    grid = np.zeros((n, n, n), np.float32)
    i0 = np.floor(p).astype(int)
    f = p - i0
    for dz in (0, 1):
        for dy in (0, 1):
            for dx in (0, 1):
                idx = i0 + np.array([dz, dy, dx])
                wt = w * np.where(dz, f[:, 0], 1 - f[:, 0]) * np.where(dy, f[:, 1], 1 - f[:, 1]) * np.where(dx, f[:, 2], 1 - f[:, 2])
                ok = np.all((idx >= 0) & (idx < n), 1)
                np.add.at(grid, tuple(idx[ok].T), wt[ok])
    grid = ndimage.gaussian_filter(grid, (sigma_A if sigma_A else voxel / 2.0) / fine)
    return grid.reshape(box, 2, box, 2, box, 2).mean((1, 3, 5)).astype(np.float32)


def zoo_map(pid, voxel="10A"):
    """(density[z, y, x], voxel size Å, source) of a zoo entry, painted from its atoms (`zoo_atoms`)."""
    atoms = zoo_atoms(pid)
    v = float(str(voxel).rstrip("A"))
    w = np.array([_Z.get(e, 6.0) for e in atoms["element"]])
    vol = atoms_to_map(atoms["xyz"], w, voxel=v)
    return vol, v, atoms_source(pid)


def prepare_map(vol: np.ndarray, size: int = 32, sigma: float = 1.0) -> np.ndarray:
    """Low-pass, centre (by centre of mass), crop/pad to ``size``³ and scale to [0, 1].

    Zoo maps are sharp sums of atoms; a Gaussian blur of ~1 voxel (1 nm)
    mimics the resolution of a cellular tomogram.
    """
    v = ndimage.gaussian_filter(np.clip(vol, 0, None).astype(np.float32), sigma)
    s = np.array(v.shape)
    src0 = np.maximum((s - size) // 2, 0)                  # where the crop starts in the input ...
    dst0 = np.maximum((size - s) // 2, 0)                  # ... and where the pad starts in the output
    n = np.minimum(s, size)
    com = np.array(ndimage.center_of_mass(v))
    shift = (size - 1) / 2 - dst0 + src0 - com             # puts the centre of mass exactly at the output's centre
    v = ndimage.shift(v, shift, order=1)
    out = np.zeros((size, size, size), np.float32)
    out[dst0[0]:dst0[0] + n[0], dst0[1]:dst0[1] + n[1], dst0[2]:dst0[2] + n[2]] = \
        v[src0[0]:src0[0] + n[0], src0[1]:src0[1] + n[1], src0[2]:src0[2] + n[2]]
    out -= out.min()
    return out / (out.max() + 1e-12)


def load_zoo(ids=("4v4r", "3j9i", "6utj", "1bxn"), size: int = 32, sigma: float = 1.0,
             verbose: bool = True) -> dict:
    """Build and prepare several zoo maps. Returns {pdb_id: volume(size³)}."""
    zoo = {}
    for pid in ids:
        raw, vs, src = zoo_map(pid)
        zoo[pid] = prepare_map(raw, size=size, sigma=sigma)
        if verbose:
            print(f"  {pid.upper():5s} {ZOO.get(pid, {}).get('name', ''):28s} raw {raw.shape} @ {vs:.0f} Å/voxel  ← {src}")
    return zoo


def random_rotation(rng: np.random.Generator) -> np.ndarray:
    """Uniformly random proper rotation (det = +1) from a random unit quaternion."""
    q = rng.normal(size=4)
    q /= np.linalg.norm(q)
    w, x, y, z = q
    return np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)]])


_P = np.array([[0, 0, 1], [0, 1, 0], [1, 0, 0]], float)  # (x,y,z) <-> (z,y,x)
def rotate_volume(vol: np.ndarray, R: np.ndarray, shift_xyz=(0, 0, 0), order: int = 1,
                  cval: float = 0.0) -> np.ndarray:
    """Rotate a volume about its centre by R (then translate by shift, in voxels, (x,y,z)).

    out(p) = vol(Rᵀ (p − c − t) + c): the *object* is rotated by R and moved by t.
    """
    R_idx = _P @ R @ _P                       # same rotation written in (z, y, x) index order
    c = (np.array(vol.shape) - 1) / 2.0
    t_idx = _P @ np.asarray(shift_xyz, float)
    M = R_idx.T                               # inverse rotation maps output -> input
    offset = c - M @ (c + t_idx)
    return ndimage.affine_transform(vol, M, offset=offset, order=order, mode="constant", cval=cval)


def rotate_image(img: np.ndarray, angle_rad: float, shift_xy=(0, 0), order: int = 1) -> np.ndarray:
    """2D analogue of :func:`rotate_volume` (counter-clockwise in the (x, y) plane)."""
    c, s = np.cos(angle_rad), np.sin(angle_rad)
    R = np.array([[c, -s], [s, c]])
    R_idx = R[::-1, ::-1]                      # (x,y) -> (y,x)
    ctr = (np.array(img.shape) - 1) / 2.0
    t_idx = np.asarray(shift_xy, float)[::-1]
    M = R_idx.T
    return ndimage.affine_transform(img, M, offset=ctr - M @ (ctr + t_idx), order=order, mode="constant")


def add_noise_snr(signal: np.ndarray, snr: float, rng: np.random.Generator) -> np.ndarray:
    """Add white Gaussian noise so that var(signal)/var(noise) = snr."""
    sigma = np.sqrt(signal.var() / snr)
    return (signal + rng.normal(0, sigma, signal.shape)).astype(np.float32)


def normalize(v: np.ndarray) -> np.ndarray:
    """Zero mean, unit standard deviation."""
    return ((v - v.mean()) / (v.std() + 1e-12)).astype(np.float32)


def pearson(a: np.ndarray, b: np.ndarray) -> float:
    """Pearson correlation of two arrays (flattened)."""
    a = a.ravel() - a.mean()
    b = b.ravel() - b.mean()
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-12))


def hungarian_accuracy(y_true, y_pred) -> float:
    """Clustering accuracy after the best one-to-one relabelling (Hungarian algorithm)."""
    from scipy.optimize import linear_sum_assignment
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    ct = np.zeros((y_pred.max() + 1, y_true.max() + 1))
    np.add.at(ct, (y_pred, y_true), 1)
    r, c = linear_sum_assignment(-ct)
    return float(ct[r, c].sum() / len(y_true))
print("🧰 toolkit loaded")

### The cast

We use the four "star" molecules of the course: the bacterial 70S **ribosome** (4V4R), the 20S **proteasome** (3J9I), the
singly **capped proteasome** (6UTJ) and **RuBisCO** (1BXN). `load_zoo` paints their density maps from the PDB models (10 Å
voxels), blurs them slightly to tomogram-like resolution, centres them in 32³ boxes and scales them to [0, 1].

In [ ]:
# ── Load the molecular zoo ──
CAST = ["4v4r", "3j9i", "6utj", "1bxn"]
NAMES = {pid: ZOO[pid]["name"] for pid in CAST}
zoo = load_zoo(CAST, size=32)                       # {pdb_id: 32×32×32 volume, 1 nm voxels}
proj = {pid: v.sum(axis=0) / v.sum(axis=0).max() for pid, v in zoo.items()}   # 2D projections along z, max = 1
print({pid: v.shape for pid, v in zoo.items()})

---
## 1 · Random variables and noise

> 📍 **Where we are.** Modules 01–03 gave you arrays, rotations and Fourier transforms — all about *clean* data. Real
> cryo-EM/ET data are dominated by noise. This module gives you the statistical tools to see through it; every later
> module (averaging, picking, classification, deep learning) relies on them.

A detector pixel does not report "the true density". It reports the true value **plus a random error** that is different
every time you look. A quantity whose value is random is a **random variable**. We describe it by its **distribution**:
which values are likely and which are rare.

🧮 **Math refresher — three summary numbers.** For $n$ measurements $x_1, \dots, x_n$ (the symbol $\sum_i$ means "add
up over all $i$"):

$$\bar x = \frac{1}{n}\sum_{i=1}^{n} x_i, \qquad
  s^2 = \frac{1}{n}\sum_{i=1}^{n}(x_i-\bar x)^2, \qquad
  s = \sqrt{s^2}.$$

*In words:* the **mean** $\bar x$ is the centre; the **variance** $s^2$ is the average squared distance from the centre;
the **standard deviation** $s$ is the typical size of a fluctuation, in the same units as $x$.

The most common noise model is the **Gaussian (normal) distribution** with mean $\mu$ and standard deviation $\sigma$:

$$p(x) = \frac{1}{\sigma\sqrt{2\pi}}\exp\!\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).$$

*In words:* values near $\mu$ are most likely; the probability falls off smoothly and symmetrically; about 68 % of values
land within $\mu\pm\sigma$ and 95 % within $\mu\pm2\sigma$. Why Gaussian? Each voxel of a tomogram is a *sum* of many
small random contributions (electron counts from dozens of tilt images, detector noise, ice), and sums of many small
independent effects become Gaussian — the **central limit theorem**. Electron *counting* itself is Poisson noise (Module 06).

In [ ]:
# ── 1.1 Draw Gaussian noise and summarise it ──
mu, sigma, n = 0.0, 1.0, 10_000
x = rng.normal(mu, sigma, n)                          # 10,000 random draws
print(f"sample mean = {x.mean():+.3f} (true {mu}),  sample std = {x.std():.3f} (true {sigma})")
print(f"fraction within ±1σ: {np.mean(np.abs(x - mu) < sigma):.3f}   within ±2σ: {np.mean(np.abs(x - mu) < 2 * sigma):.3f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.2))
a1.hist(x, bins=60, density=True, color="lightgray", edgecolor="gray", label="10,000 samples")
g = np.linspace(-4, 4, 200)
a1.plot(g, stats.norm.pdf(g, mu, sigma), "tab:red", lw=2, label="Gaussian formula")
a1.set_xlabel("value (arbitrary units)"); a1.set_ylabel("probability density"); a1.legend(fontsize=7)
a1.set_title("a histogram approaches the distribution")
running = np.cumsum(x) / np.arange(1, n + 1)          # mean of the first k samples
k = np.arange(1, n + 1)
a2.plot(k, running, color="tab:blue", lw=1, label="running mean")
a2.plot(k, 1 / np.sqrt(k), "k--", lw=1, label=r"$\pm\sigma/\sqrt{k}$")
a2.plot(k, -1 / np.sqrt(k), "k--", lw=1)
a2.set_xscale("log"); a2.set_ylim(-1.5, 1.5)
a2.set_xlabel("number of samples k (log scale)"); a2.set_ylabel("mean of first k samples")
a2.set_title("law of large numbers"); a2.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The histogram of many random draws traces the Gaussian curve, and the running mean settles
on the true mean — but slowly: its error shrinks like $\sigma/\sqrt{k}$ (dashed lines). To halve the uncertainty you need
**four times** as many samples. Remember this $1/\sqrt{k}$: it returns in Section 4 as the reason structural biologists
average thousands of particles.

---
## 2 · Similarity: the Pearson correlation coefficient

Almost every algorithm in this course must answer: *how similar are these two images (or volumes)?* Template matching
(Module 11), alignment (Modules 12, 21), classification (Module 14) and resolution (Module 10) all use the same score, the
**Pearson correlation coefficient**:

$$r(a,b) = \frac{\sum_i (a_i-\bar a)(b_i-\bar b)}{\sqrt{\sum_i (a_i-\bar a)^2}\;\sqrt{\sum_i (b_i-\bar b)^2}}.$$

*In words:* subtract each image's mean, multiply the two images voxel by voxel, add up, and divide by the two "sizes".
$r=1$ means identical up to brightness and contrast, $r=0$ means unrelated, $r=-1$ means inverted. Because means and
scales are divided out, $r$ ignores changes in **brightness** (adding a constant) and **contrast** (multiplying by a
positive constant) — exactly the nuisances that vary between micrographs.

In [ ]:
# ── 2.1 What does the correlation score respond to? ──
ribo = proj["4v4r"]
tests = {
    "same ribosome,\nbrightness+contrast changed": 3.0 * ribo + 7.0,
    "same ribosome,\nrotated 30°": rotate_image(ribo, np.radians(30)),
    "20S proteasome\n(different molecule)": proj["3j9i"],
    "ribosome + noise\n(SNR 0.1)": add_noise_snr(ribo, 0.1, rng),
}
fig, axes = plt.subplots(1, 5, figsize=(10, 2.5))
axes[0].imshow(ribo); axes[0].set_title("reference:\n70S ribosome projection", fontsize=8)
for ax, (lab, img) in zip(axes[1:], tests.items()):
    ax.imshow(img); ax.set_title(f"{lab}\nr = {pearson(ribo, img):+.2f}", fontsize=8)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Brightness and contrast changes leave $r$ at exactly 1. A rotation lowers it (so we must
*align* before comparing — Module 12). A different molecule lowers it only a little: most of the score comes from
"a bright object in the middle of an empty box", which every molecule shares — one reason template matching needs care
(Module 11). **Noise lowers $r$ the most.** In the next section we turn that observation into a way of *measuring* noise.

---
## 3 · Signal-to-noise ratio, two ways

### 3.1 The variance ratio (the course's simulation convention)

Write a measured image as signal plus noise, $x = s + n$. The **signal-to-noise ratio** used throughout this course (and
by the toolkit's `add_noise_snr`) is

$$\mathrm{SNR} = \frac{\operatorname{var}(s)}{\operatorname{var}(n)}.$$

*In words:* how much the molecule makes the pixel values vary, compared with how much the noise does. SNR = 1 means
equal; SNR = 0.01 means the noise variance is 100× larger (its standard deviation 10× larger). Engineers often quote
decibels, $10\log_{10}\mathrm{SNR}$: SNR 0.01 = −20 dB.

In [ ]:
# ── 3.1 One ribosome projection at decreasing SNR ──
levels = [np.inf, 1.0, 0.1, 0.03, 0.01]
fig, axes = plt.subplots(1, len(levels), figsize=(10, 2.3))
for ax, snr in zip(axes, levels):
    img = ribo if np.isinf(snr) else add_noise_snr(ribo, snr, rng)
    ax.imshow(img); ax.axis("off")
    ax.set_title("noise-free" if np.isinf(snr) else f"SNR {snr:g}  ({10 * np.log10(snr):+.0f} dB)", fontsize=8)
fig.suptitle("70S ribosome projection (32 × 32 nm) — cellular cryo-ET lives on the right", fontsize=9)
plt.tight_layout(); plt.show()

### 3.2 Measuring SNR when you do not know the signal

In real data we never see $s$ alone, so how can anyone quote an SNR? Take **two independent noisy copies** of the same
signal, $x_1 = s + n_1$ and $x_2 = s + n_2$, with the same noise level. Their only shared ingredient is $s$, so their
covariance is $\operatorname{var}(s)$, while each has total variance $\operatorname{var}(s) + \operatorname{var}(n)$.
Their correlation is therefore

$$c = \frac{\operatorname{var}(s)}{\operatorname{var}(s)+\operatorname{var}(n)} = \frac{\mathrm{SNR}}{1+\mathrm{SNR}}
\quad\Longleftrightarrow\quad \boxed{\mathrm{SNR} = \frac{c}{1-c}}.$$

*In words:* the more two noisy copies resemble each other, the more signal they contain. This trick goes back to Frank &
Al-Ali (1975), and it is how the **BOE-ViT** paper (Module 21) defines the SNR of its simulated test sets
($\mathrm{SNR}=c/(1-c)$ from the correlation $c$ of aligned pairs). The same "two independent halves" idea underlies the
Fourier shell correlation (FSC) used to measure resolution in Module 10.

In [ ]:
# ── 3.2 Estimate SNR from pairs of noisy subtomograms ──
def snr_from_pair(x1: np.ndarray, x2: np.ndarray) -> float:
    """SNR estimated from two aligned noisy copies of the same signal: c / (1 − c)."""
    c = pearson(x1, x2)
    return c / (1 - c)


ribo3d = zoo["4v4r"]
true_snrs = [3.0, 1.0, 0.3, 0.1, 0.03, 0.01]
n_pairs = 60
estimates = {}
for snr in true_snrs:
    estimates[snr] = np.array([snr_from_pair(add_noise_snr(ribo3d, snr, rng), add_noise_snr(ribo3d, snr, rng))
                               for _ in range(n_pairs)])
    e = estimates[snr]
    print(f"true SNR {snr:5.2f}: median estimate {np.median(e):.4f}, middle 90 % of single pairs "
          f"[{np.percentile(e, 5):+.4f}, {np.percentile(e, 95):+.4f}]")

fig, ax = plt.subplots(figsize=(5, 3.4))
for snr in true_snrs:
    e = estimates[snr]
    ax.scatter(np.full(n_pairs, snr) * np.exp(rng.normal(0, 0.05, n_pairs)), np.clip(e, 1e-4, None),
               s=6, alpha=0.4, color="tab:blue")
    ax.scatter(snr, np.median(e), marker="_", s=300, color="tab:red", lw=2)
ax.plot([0.005, 5], [0.005, 5], "k--", lw=1, label="estimate = truth")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("true SNR (variance ratio)"); ax.set_ylabel("estimated SNR = c/(1−c)")
ax.set_title("SNR from pairs of 32³ ribosome subtomograms\n(dots: single pairs; red: median)")
ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** The median estimate lands on the diagonal: $c/(1-c)$ recovers the variance-ratio SNR without
ever seeing the clean signal. But a *single* pair is unreliable at low SNR — at SNR 0.01 single estimates scatter widely
(some are even negative and are drawn at the bottom of the plot). Even *measuring* how noisy the data are requires many
measurements.

✏️ **Your turn.** Replace the 32³ volume `ribo3d` by the 32 × 32 projection `ribo` (≈ 32× fewer pixels) and re-run.
How much wider does the spread become, and why?

---
## 4 · Averaging: the √N law

If noise is what hides the molecule, the cure is **averaging many copies**. Suppose $N$ copies are perfectly aligned:
$x_j = s + n_j$ with independent noise of variance $\sigma^2$. Their average is

$$\bar x_N = s + \frac{1}{N}\sum_{j=1}^{N} n_j, \qquad
  \operatorname{var}\Big(\frac{1}{N}\sum_j n_j\Big) = \frac{\sigma^2}{N}
  \;\Rightarrow\; \mathrm{SNR}_N = N\cdot\mathrm{SNR}_1 .$$

*In words:* the signal is the same in every copy and survives averaging; the noise points in random directions and
partly cancels. The noise **standard deviation** shrinks like $1/\sqrt N$ and the SNR grows like $N$. This single fact is
why single-particle cryo-EM averages $10^4$–$10^6$ particle images and why subtomogram averaging needs thousands of copies.

In [ ]:
# ── 4.1 Average N aligned noisy copies of the ribosome projection ──
snr1 = 0.05                                           # per-copy SNR, typical of a cellular subtomogram
Ns = [1, 4, 16, 64, 256, 1024]
copies = np.stack([add_noise_snr(ribo, snr1, rng) for _ in range(max(Ns))])   # 1024 aligned copies
avgs = {N: copies[:N].mean(axis=0) for N in Ns}
noise_std = np.array([np.std(avgs[N] - ribo) for N in Ns])
snr_meas = np.array([ribo.var() / np.var(avgs[N] - ribo) for N in Ns])
for N, sd, sn in zip(Ns, noise_std, snr_meas):
    print(f"N = {N:5d}: residual noise std {sd:6.3f}   measured SNR {sn:7.2f}   (theory {N * snr1:6.2f})")

fig = plt.figure(figsize=(10, 4.2))
for i, N in enumerate(Ns):
    ax = fig.add_subplot(2, len(Ns), i + 1)
    ax.imshow(avgs[N]); ax.axis("off"); ax.set_title(f"N = {N}\nr = {pearson(avgs[N], ribo):.2f}", fontsize=8)
ax = fig.add_subplot(2, 2, 3)
ax.loglog(Ns, noise_std, "o-", label="measured"); ax.loglog(Ns, noise_std[0] / np.sqrt(Ns), "k--", label=r"$\propto 1/\sqrt{N}$")
ax.set_xlabel("number of averaged copies N"); ax.set_ylabel("noise std"); ax.legend(fontsize=7)
ax = fig.add_subplot(2, 2, 4)
ax.loglog(Ns, snr_meas, "o-", color="tab:green", label="measured"); ax.loglog(Ns, snr1 * np.array(Ns), "k--", label=r"$N\cdot\mathrm{SNR}_1$")
ax.set_xlabel("number of averaged copies N"); ax.set_ylabel("SNR of the average"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** One copy at SNR 0.05 is unrecognisable; 64 copies already show the ribosome's outline;
1,024 copies give a nearly clean image. The measured points sit on the theoretical lines: noise std ∝ $1/\sqrt N$,
SNR ∝ $N$. The correlation $r$ with the truth in each title climbs towards 1.

### 4.2 ⚠️ The catch: copies must be aligned

In a cell, every molecule points in a different direction. What happens if we average copies that are **not** aligned?
We use the 20S proteasome seen from the side (an elongated barrel), so that the effect is easy to see.

In [ ]:
# ── 4.2 Averaging copies with random in-plane orientations ──
barrel = proj["3j9i"]
aligned = np.mean([add_noise_snr(barrel, snr1, rng) for _ in range(256)], axis=0)
angles = rng.uniform(0, 2 * np.pi, 256)                  # each particle lies at its own angle in the ice
misaligned = np.mean([add_noise_snr(rotate_image(barrel, a), snr1, rng) for a in angles], axis=0)
fig, axes = plt.subplots(1, 3, figsize=(7, 2.6))
for ax, img, t in zip(axes, [barrel, aligned, misaligned],
                      ["20S proteasome (truth)", "256 aligned copies", "256 randomly rotated copies"]):
    ax.imshow(img); ax.axis("off"); ax.set_title(f"{t}\nr = {pearson(img, barrel):.2f}", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Averaging *misaligned* copies also removes the noise — but it removes the molecule's shape
with it: the barrel becomes a round smudge (its rotational average) and the correlation with the truth drops. Averaging only helps **after alignment** (Modules 12, 13, 21),
and only if the copies are the **same species in the same state** — which is why we need *classification* (Sections 7–10
of this module, and Modules 14–23). Alignment and classification are chicken-and-egg problems; the lab's methods (FAML,
Multi-Pattern Pursuit, DISCA) solve them *jointly* and iteratively.

---
## 5 · Hypothesis testing: is a difference real, or just noise?

Biology papers constantly ask questions like *"does the drug change how proteins are arranged?"* or *"are these two
averages genuinely different structures?"*. Because data are noisy, even two groups drawn from the *same* population
differ a little. A **hypothesis test** quantifies how surprising the observed difference would be if there were no real effect.

| Term | Meaning |
|---|---|
| **Null hypothesis $H_0$** | "no real effect" (e.g., drug and control membranes have the same spacing distribution) |
| **Test statistic $T$** | a number summarising the effect (difference of means, a rank sum, …) |
| **Null distribution** | the distribution of $T$ if $H_0$ were true |
| **p-value** | probability, under $H_0$, of a statistic at least as extreme as the one observed |
| **Significance level α** | threshold chosen *beforehand* (0.05; Multi-Pattern Pursuit uses 0.01) |
| **Type I / Type II error** | rejecting a true $H_0$ (false alarm) / missing a real effect |

⚠️ A p-value is **not** the probability that $H_0$ is true, and a tiny p-value does not mean a *large* effect.

### 5.1 A toy membrane: Pma1 hexamers with and without a drug

🧫 In the plasma membrane of the pathogenic yeast *Candida glabrata*, the proton pump **Pma1** forms hexamers (≈ 16 nm
rosettes) packed into clusters. Cryo-ET of isolated membranes showed that the antifungal drug **caspofungin** disperses
these clusters (Jiang *et al.* 2025; paper box below). We simulate a **toy** version: each "membrane patch" is a
500 × 500 nm square with clusters of hexamers at a typical spacing of 16.5 nm (control), or spread out ≈ 1.7× (drug),
or ≈ 1.5× (a mutant, Δ*fen1*, whose membranes the paper reports are already dispersed). For each hexamer we measure the
**nearest-neighbour distance** (NND).

In [ ]:
# ── 5.1 Simulate toy membrane patches of Pma1 hexamers ──
rng = np.random.default_rng(SEED + 51)    # re-seed: results here do not depend on how many random numbers earlier cells used
def simulate_membrane(rng, spacing_nm: float, n_clusters: int = 10, patch_nm: float = 500.0,
                      n_single: int = 25) -> tuple[np.ndarray, np.ndarray]:
    """Toy 2D point pattern: clusters of hexamers (hard-core distance 0.9·spacing) plus isolated hexamers.

    Returns (positions in nm [n, 2], nearest-neighbour distance of every hexamer in nm).
    """
    pts = []

    def free(p):
        return all(np.hypot(*(p - q)) > 0.9 * spacing_nm for q in pts)
    for _ in range(n_clusters):
        centre = rng.uniform(40, patch_nm - 40, 2)
        n_members, placed, tries = 4 + rng.poisson(6), 0, 0
        while placed < n_members and tries < 500:
            tries += 1
            p = centre + rng.normal(0, spacing_nm * np.sqrt(n_members) / 2.5, 2)
            if free(p):
                pts.append(p); placed += 1
    for _ in range(n_single):
        for _ in range(100):
            p = rng.uniform(0, patch_nm, 2)
            if free(p):
                pts.append(p); break
    pts = np.array(pts)
    dist, _ = cKDTree(pts).query(pts, k=2)          # k=2: the closest point is the hexamer itself
    return pts, dist[:, 1]


CONDITIONS = {"control": 16.5, "caspofungin": 16.5 * 1.7, "Δfen1": 16.5 * 1.5}   # toy spacings (nm)
N_PATCHES = 6                                                                      # membrane patches per condition
membranes = {cond: [simulate_membrane(rng, s) for _ in range(N_PATCHES)] for cond, s in CONDITIONS.items()}
patch_medians = {cond: np.array([np.median(d) for _, d in pats]) for cond, pats in membranes.items()}
for cond, med in patch_medians.items():
    n_hex = sum(len(p) for p, _ in membranes[cond])
    print(f"{cond:12s}: {n_hex:4d} hexamers in {N_PATCHES} patches; per-patch median NND (nm): {np.round(med, 1)}")

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, cond in zip(axes[:2], ["control", "caspofungin"]):
    pts, _ = membranes[cond][0]
    ax.scatter(pts[:, 0], pts[:, 1], s=9, color="tab:purple")
    ax.set_aspect("equal"); ax.set_xlim(0, 500); ax.set_ylim(0, 500)
    ax.set_title(f"toy patch — {cond}", fontsize=9); ax.set_xlabel("x (nm)"); ax.set_ylabel("y (nm)")
for cond, col in zip(CONDITIONS, ["tab:blue", "tab:red", "tab:orange"]):
    allnnd = np.concatenate([d for _, d in membranes[cond]])
    axes[2].hist(allnnd, bins=np.arange(10, 80, 2.5), histtype="step", lw=1.5, color=col, label=cond, density=True)
axes[2].set_xlabel("nearest-neighbour distance (nm)"); axes[2].set_ylabel("density"); axes[2].legend(fontsize=7)
axes[2].set_title("NND distributions are skewed", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** The drug-like patches look sparser and their NND distribution is shifted right and is
**skewed** (a long right tail from isolated hexamers) — not a Gaussian bell. Tests that assume Gaussian data are risky
here; we use tests that need no such assumption.

### 5.2 The permutation test: build the null distribution yourself

**Idea.** If the drug had no effect, the labels "control" and "caspofungin" would be arbitrary: any reshuffling of the
12 patch medians into two groups of 6 would be as plausible as the real one. So we compute the test statistic — the
difference of group means, $T = \bar m_{\text{drug}} - \bar m_{\text{control}}$ — for **every** possible relabelling
($\binom{12}{6} = 924$ of them) and ask what fraction is at least as extreme as the observed value:

$$p = \frac{\#\{\text{relabellings with } |T_\pi| \ge |T_{\text{obs}}|\}}{\#\{\text{relabellings}\}}.$$

*In words:* the null distribution comes from the data themselves; no Gaussian assumption is needed. We use one median per
patch because patches — not individual hexamers — are the independent samples (more on this in 5.3).

In [ ]:
# ── 5.2 Exact permutation test on per-patch medians (control vs caspofungin) ──
def permutation_test_exact(a: np.ndarray, b: np.ndarray):
    """Two-sided exact permutation test for a difference in means. Returns (T_obs, all T_perm, p)."""
    pooled = np.concatenate([a, b])
    t_obs = b.mean() - a.mean()
    t_perm = []
    for idx_b in itertools.combinations(range(len(pooled)), len(b)):   # every way to choose the "b" group
        mask = np.zeros(len(pooled), bool); mask[list(idx_b)] = True
        t_perm.append(pooled[mask].mean() - pooled[~mask].mean())
    t_perm = np.array(t_perm)
    p = np.mean(np.abs(t_perm) >= abs(t_obs) - 1e-12)
    return t_obs, t_perm, p


t_obs, t_perm, p_perm = permutation_test_exact(patch_medians["control"], patch_medians["caspofungin"])
print(f"observed difference of mean patch medians: {t_obs:+.2f} nm;  {len(t_perm)} relabellings;  p = {p_perm:.4f}")
print(f"smallest possible p with 6 vs 6 patches: 2/924 = {2 / 924:.4f}")
fig, ax = plt.subplots(figsize=(5.5, 3))
ax.hist(t_perm, bins=40, color="lightgray", edgecolor="gray", label="null: all 924 relabellings")
ax.axvline(t_obs, color="tab:red", lw=2, label=f"observed {t_obs:+.1f} nm")
ax.set_xlabel("difference of mean patch medians (nm)"); ax.set_ylabel("count"); ax.legend(fontsize=7)
ax.set_title("exact permutation test, control vs caspofungin (toy)"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Only the two most extreme relabellings — the real one and its mirror image — reach the observed
difference, so $p = 2/924 \approx 0.002$, the smallest p-value 6-vs-6 patches can ever give. Notice the floor: **with few
independent samples, no test can give a very small p-value**, however large the effect.

### 5.3 More than two groups: the Kruskal–Wallis test (+ Dunn's post-hoc)

With three conditions we could run three pairwise tests, but each extra test is another chance of a false alarm.
The **Kruskal–Wallis test** asks one question first: *do all groups come from the same distribution?* It works on
**ranks**: pool all $N$ values, sort them, replace each value by its rank $1, \dots, N$, and compare the mean rank
$\bar R_g$ of each group $g$ (size $n_g$) with the overall mean rank $(N+1)/2$:

$$H = \frac{12}{N(N+1)}\sum_{g} n_g\left(\bar R_g - \frac{N+1}{2}\right)^2 .$$

*In words:* if the groups are alike, every group's average rank is near the middle and $H$ is small. Under $H_0$, $H$
approximately follows a **χ² distribution** with (number of groups − 1) degrees of freedom, which gives the p-value.
Ranks make the test robust to skew and outliers. If $H$ is significant, **Dunn's test** compares pairs of groups by their
mean-rank differences, $z_{gh} = (\bar R_g-\bar R_h)\big/\sqrt{\tfrac{N(N+1)}{12}\left(\tfrac1{n_g}+\tfrac1{n_h}\right)}$,
with a **Bonferroni** correction (multiply each p-value by the number of comparisons).

In [ ]:
# ── 5.3 Kruskal–Wallis and Dunn by hand, checked against SciPy ──
def kruskal_wallis(groups: list) -> tuple[float, float, np.ndarray]:
    """Kruskal–Wallis H (no tie correction) and its χ² p-value. Returns (H, p, mean rank per group)."""
    pooled = np.concatenate(groups)
    ranks = stats.rankdata(pooled)                       # 1 = smallest value
    N = len(pooled)
    bounds = np.cumsum([0] + [len(g) for g in groups])
    mean_ranks = np.array([ranks[bounds[i]:bounds[i + 1]].mean() for i in range(len(groups))])
    H = 12 / (N * (N + 1)) * sum(len(g) * (mr - (N + 1) / 2) ** 2 for g, mr in zip(groups, mean_ranks))
    return H, stats.chi2.sf(H, df=len(groups) - 1), mean_ranks


def dunn_test(groups: list, names: list) -> None:
    """Print Dunn's pairwise z statistics with Bonferroni-corrected two-sided p-values."""
    _, _, mr = kruskal_wallis(groups)
    N = sum(len(g) for g in groups)
    pairs = list(itertools.combinations(range(len(groups)), 2))
    for i, j in pairs:
        z = (mr[i] - mr[j]) / np.sqrt(N * (N + 1) / 12 * (1 / len(groups[i]) + 1 / len(groups[j])))
        p = min(1.0, 2 * stats.norm.sf(abs(z)) * len(pairs))
        print(f"   Dunn {names[i]:>11s} vs {names[j]:<11s}: z = {z:+.2f}, Bonferroni p = {p:.4f}")


names = list(CONDITIONS)
groups = [patch_medians[c] for c in names]
H, p_kw, mean_ranks = kruskal_wallis(groups)
H_sp, p_sp = stats.kruskal(*groups)
print(f"per-PATCH medians (n = {N_PATCHES} per group): H = {H:.2f}, p = {p_kw:.5f}   | SciPy: H = {H_sp:.2f}, p = {p_sp:.5f}")
print("   mean ranks:", dict(zip(names, np.round(mean_ranks, 1))))
dunn_test(groups, names)
pooled_groups = [np.concatenate([d for _, d in membranes[c]]) for c in names]
H_all, p_all = stats.kruskal(*pooled_groups)
print(f"per-HEXAMER values (n ≈ {len(pooled_groups[0])} per group): H = {H_all:.1f}, p = {p_all:.1e}   ← pseudo-replication!")

fig, ax = plt.subplots(figsize=(5, 3))
for i, c in enumerate(names):
    ax.scatter(np.full(N_PATCHES, i) + rng.uniform(-0.08, 0.08, N_PATCHES), patch_medians[c], s=25)
    ax.hlines(np.median(patch_medians[c]), i - 0.25, i + 0.25, color="k")
ax.set_xticks(range(3)); ax.set_xticklabels(names); ax.set_ylabel("median NND per patch (nm)")
ax.set_title(f"Kruskal–Wallis on patch medians: H = {H:.1f}, p = {p_kw:.4f}"); plt.tight_layout(); plt.show()

🔍 **What just happened?** Our hand-written $H$ matches SciPy exactly (with continuous data there are no ties to
correct for). The groups differ (p < 0.001). Dunn's test confirms control vs caspofungin, but — surprisingly — calls
the intermediate Δfen1 group *not significantly* different from either neighbour after the Bonferroni correction, even
though all six Δfen1 patches lie above all six control patches. Rank tests on 6 + 6 + 6 values are **conservative**:
with so few replicates, only the largest contrast survives. That is the honest price of a small experiment.

⚠️ **Pseudo-replication.** Running the same test on the 700+ individual hexamers per group gives an absurd
$p < 10^{-100}$. The hexamers within one membrane are *not* independent experiments — they share the same cell,
preparation and tomogram. The **unit of replication** (here: the membrane patch, ideally independent biological
replicates) determines what a p-value means. The same idea reappears as **data leakage** in machine learning (Module 05):
particles from one tomogram must not be split between training and test sets.

### 📄 Paper connection — the fungal plasma membrane

**Paper.** Jiang J, Keniya MV, Puri A, Zhan X, Cheng J, Wang H, Lin G, Lee Y-K, Jaber N, Zhao C, Pang C, Hassoun Y,
Zheng H, Shor E, Shi Z, Lee S-H, **Xu M**, Perlin DS, Dai W (2025). *Molecular landscape of the fungal plasma membrane and
implications for antifungal action.* **Nature Communications** 16:9125. [doi:10.1038/s41467-025-64171-x](https://doi.org/10.1038/s41467-025-64171-x)

- **Question it answers.** How are the large protein complexes of a pathogenic yeast's plasma membrane organised in the
  native membrane, and how does the echinocandin drug caspofungin — which targets β-(1,3)-glucan synthase (Fks1) — change it?
- **What the paper actually does.** Cryo-ET mainly of plasma membranes isolated from *Candida glabrata* spheroplasts
  (plus cryo-FIB-milled lamellae of intact cells), with features annotated by EMAN2's CNN-based tomogram annotation and
  manually curated; subtomogram averaging of glucan synthase; quantitative proteomics; micropipette measurements of
  membrane tension. The spatial analysis uses Pma1 **nearest-neighbour distances**, defines clusters as **≥ 4 hexamers at
  ≈ 160–170 Å spacing**, checks them with **Gaussian mixture models (AIC/BIC) and k-means (inertia)**, and compares
  conditions with **Kruskal–Wallis and Dunn tests** — the toolbox of this module.
- **What our toy keeps.** Skewed NND distributions, rank-based tests for several conditions, post-hoc pairwise
  comparisons, and the danger of pseudo-replication.
- **What our toy leaves out.** Real tomograms, particle annotation, the cluster definition, glucan synthase, the
  proteomics and membrane-tension experiments. Our spacings and patch counts are invented.
- **Result reported.** The authors report Pma1 hexamers in dense clusters covering ≈ 15 % of the membrane; caspofungin
  increases intra-cluster distance and cluster radius about 1.7× in wild type and lowers membrane tension (≈ 2.6× in wild
  type); Δ*fen1* membranes are already dispersed and less drug-susceptible — consistent with a lipid-mediated mechanism.
- **Xu-lab role.** Min Xu and Xueying Zhan (CMU) are co-authors; the paper does not itemise which computational step
  they contributed, so we do not claim one. **Code:** no code statement found; tomograms are EMD-45105 / EMD-45106.
  Module 33 revisits this study in depth.

### 5.4 Paired data: the Wilcoxon signed-rank test (Multi-Pattern Pursuit's redundancy check)

🧫 **Setting.** A pattern-mining algorithm has produced two candidate averages, **A** and **B**. Are they two different
complexes, or did the algorithm split one complex into two redundant patterns? Multi-Pattern Pursuit (Xu *et al.* 2019)
answers with a test: take the particles assigned to A, score each one against A **and** against B (B being the pattern
with the better quality score), and ask whether A's members *significantly prefer* A. If not, A is redundant and dropped.

The scores are **paired** — the same particle scored twice — so we analyse the differences $d_i = c_{iA} - c_{iB}$.
The **Wilcoxon signed-rank test**: rank the $|d_i|$ from smallest (rank 1) to largest, and add up the ranks of the
*positive* differences:

$$W^{+} = \sum_{i:\,d_i>0} \operatorname{rank}(|d_i|).$$

*In words:* if particles truly prefer A, the positive differences will be both more numerous and larger, so $W^{+}$ is
big. Under $H_0$ (no preference) every sign is a coin flip, so we can build the null distribution by **randomly flipping
signs**. We use one-sided significance α = 0.01, as in the paper.

**Two scenarios** with aligned 3D subtomograms (SNR 0.2): (i) *redundant* — A and B are both averages of 20S
proteasomes (B from more particles); (ii) *distinct* — A is the 20S proteasome, B the capped proteasome.
⚠️ We score **fresh** particles assigned to A, not the ones used to build A: a particle's own noise inside the average
would otherwise bias the test toward A ("double dipping", see Module 10).

In [ ]:
# ── 5.4 Wilcoxon signed-rank test, by hand and with SciPy ──
rng = np.random.default_rng(SEED + 54)
def noisy_copies(pid: str, n: int, snr: float) -> np.ndarray:
    """n aligned noisy copies of a zoo volume (already-aligned subtomograms)."""
    return np.stack([add_noise_snr(zoo[pid], snr, rng) for _ in range(n)])


def wilcoxon_signed_rank(d: np.ndarray, n_flips: int = 20_000):
    """One-sided (d > 0) signed-rank test. Returns (W+, null samples from random sign flips, p)."""
    ranks = stats.rankdata(np.abs(d))
    w_plus = ranks[d > 0].sum()
    signs = rng.random((n_flips, len(d))) < 0.5            # each sign is a fair coin under H0
    w_null = (signs * ranks).sum(axis=1)
    p = (np.sum(w_null >= w_plus) + 1) / (n_flips + 1)
    return w_plus, w_null, p


SNR_P = 0.2
pattern_A = noisy_copies("3j9i", 40, SNR_P).mean(axis=0)
candidates_B = {"redundant (B = 20S)": noisy_copies("3j9i", 80, SNR_P).mean(axis=0),
                "distinct (B = capped)": noisy_copies("6utj", 80, SNR_P).mean(axis=0)}
members_A = noisy_copies("3j9i", 30, SNR_P)              # fresh particles assigned to A
results = {}
for case, pattern_B in candidates_B.items():
    d = np.array([pearson(m, pattern_A) - pearson(m, pattern_B) for m in members_A])
    w_plus, w_null, p_hand = wilcoxon_signed_rank(d)
    sp = stats.wilcoxon(d, alternative="greater")
    results[case] = (d, w_plus, w_null)
    verdict = "keep both patterns" if sp.pvalue < 0.01 else "A is redundant → drop A"
    print(f"{case:22s}: mean d = {d.mean():+.4f}, W+ = {w_plus:.0f} (SciPy {sp.statistic:.0f}), "
          f"p by sign flips = {p_hand:.2e}, SciPy p = {sp.pvalue:.2e} → {verdict}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3))
for i, (case, (d, w_plus, w_null)) in enumerate(results.items()):
    axes[0].scatter(np.full(len(d), i) + rng.uniform(-0.1, 0.1, len(d)), d, s=12)
    axes[1].hist(w_null, bins=40, histtype="step", lw=1.5, color=f"C{i}", label=f"null ({case})")
    axes[1].axvline(w_plus, color=f"C{i}", ls="--", lw=2)
axes[0].axhline(0, color="k", lw=0.8); axes[0].set_xticks([0, 1]); axes[0].set_xticklabels(list(results), fontsize=8)
axes[0].set_ylabel(r"$d_i = c_{iA} - c_{iB}$"); axes[0].set_title("paired score differences of A's members")
axes[1].set_xlabel("W+ (dashed: observed)"); axes[1].set_ylabel("count"); axes[1].legend(fontsize=7)
axes[1].set_title("sign-flip null distributions")
plt.tight_layout(); plt.show()

🔍 **What just happened?** In the *redundant* case A's members actually correlate slightly *better* with B (it averages
more particles, so it is less noisy), $W^{+}$ is small and the test says "no significant preference": A would be dropped.
In the *distinct* case nearly every particle prefers A, $W^{+}$ sits far in the tail of the null, and both patterns are
kept — the test "sees" the proteasome's missing cap even though each particle's correlation difference is tiny. Our
sign-flip p-value agrees with SciPy's exact p-value (the Monte Carlo version cannot go below ≈ 1/20,000).

### 📄 Paper connection — Multi-Pattern Pursuit (de novo structural pattern mining)

**Paper.** Xu M, Singla J, Tocheva EI, Chang Y-W, Stevens RC, Jensen GJ, Alber F (2019). *De novo structural pattern
mining in cellular electron cryotomograms.* **Structure** 27(4):679–691.e14. [doi:10.1016/j.str.2019.01.005](https://doi.org/10.1016/j.str.2019.01.005).
Preprint: arXiv:1512.09347 (titled *"De novo visual proteomics in single cells through pattern mining"*).

- **Question it answers.** Can we find *all* abundant recurring structures ("patterns") in tomograms of cells without
  templates, without knowing how many there are?
- **What the paper actually does.** The Multi-Pattern Pursuit (MPP) loop: generate candidate patterns (k-means on
  features of aligned, wedge-imputed subtomograms), score each average by an SSNR-based quality score (SFSC), greedily
  select good non-overlapping patterns, re-align every subtomogram to every pattern, and **remove redundant patterns with a
  Wilcoxon signed-rank test at significance 0.01**; iterate until the selection is stable.
- **What our toy keeps.** The redundancy test itself: paired alignment scores of A's members against A and B, one-sided
  signed-rank test, drop A if it is not significantly preferred.
- **What our toy leaves out.** Alignment (our copies are pre-aligned), missing wedges, SFSC, k-means pattern generation
  and the iterative loop — the full method is Module 15.
- **Result reported (arXiv v3).** On simulated data, 12 patterns after 32 iterations, 8 matching unique complexes at
  ≤ 10 % false-discovery rate; complexes with fewer than ≈ 200–300 copies were often missed. On ≈ 30,000 subtomograms from
  *Acetonema longum*, *Hylemonella gracilis* and *Bdellovibrio bacteriovorus* the authors report 12, 15 and 10 patterns,
  including ribosome-like and membrane-associated patterns.
- **Code.** AITom `aitom/tomominer/pursuit/multi/` (original: github.com/alberlab/tomominer).

---
## 6 · Likelihood: which parameters make the data most probable?

Statistics has a second, very powerful way of reasoning: write down a **model** of how the data were generated, then
find the parameters under which the observed data would have been **most probable**. For independent measurements the
probabilities multiply, and it is easier to add logarithms:

$$\mathcal L(\theta) = \prod_{i=1}^{n} p(x_i\mid\theta), \qquad
  \ell(\theta) = \log\mathcal L(\theta) = \sum_{i=1}^{n}\log p(x_i\mid\theta).$$

*In words:* the **likelihood** $\mathcal L(\theta)$ scores how well parameters $\theta$ explain the data; the
**maximum-likelihood estimate (MLE)** is the $\theta$ with the highest score. For Gaussian data the MLE of $\mu$ is the
sample mean and the MLE of $\sigma$ is the sample standard deviation (with $1/n$). "ML" in FAML, RELION and many lab
papers means exactly this.

In [ ]:
# ── 6.1 Likelihood of the noise level in an empty background region ──
background = rng.normal(0.0, 2.0, 500)                    # 500 voxels of pure noise, true σ = 2
print("product of 500 probabilities:", np.prod(stats.norm.pdf(background, 0, 2.0)), "← underflows to 0, hence logs")
sig_grid = np.linspace(1.0, 4.0, 300)
loglik = np.array([stats.norm.logpdf(background, background.mean(), s).sum() for s in sig_grid])
s_mle = sig_grid[np.argmax(loglik)]
print(f"MLE of σ on the grid: {s_mle:.3f};  sample std (1/n): {background.std():.3f};  truth: 2.0")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(sig_grid, loglik); ax.axvline(s_mle, color="tab:red", ls="--", label="maximum")
ax.set_xlabel("noise level σ (model parameter)"); ax.set_ylabel("log-likelihood ℓ(σ)")
ax.set_title("maximum likelihood estimate of the noise level"); ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** Multiplying 500 small probabilities gives a number too small for the computer (0.0), which is
why everyone works with **log**-likelihoods. The log-likelihood curve peaks at the sample standard deviation — the MLE.
Next: what if part of the model is **hidden**, such as *which molecule* each particle is?

---
## 7 · Mixture models and the EM algorithm

### 7.1 One dimension: two populations in one histogram

🧫 Suppose we measure a size-like feature for 300 particles in a tomogram. Two species are mixed — say RuBisCO
(≈ 11 nm) and something larger (≈ 15 nm) — but nobody told us which particle is which. A **Gaussian mixture model
(GMM)** says each particle first "chooses" a hidden class $z_i = k$ with probability $\pi_k$, then draws its feature
from that class's Gaussian:

$$p(x) = \sum_{k=1}^{K} \pi_k\,\mathcal N(x\mid\mu_k,\sigma_k^2), \qquad \sum_k \pi_k = 1 .$$

If we knew the labels we would just compute each class's mean and variance; if we knew the parameters we could compute
the labels. The **expectation–maximization (EM)** algorithm (Dempster, Laird & Rubin 1977) alternates between the two:

- **E-step** — soft labels ("responsibilities"): for every particle, the probability it belongs to class $k$:
  $$\gamma_{ik} = \frac{\pi_k\,\mathcal N(x_i\mid\mu_k,\sigma_k^2)}{\sum_{j}\pi_j\,\mathcal N(x_i\mid\mu_j,\sigma_j^2)}.$$
- **M-step** — weighted maximum likelihood, with $N_k = \sum_i\gamma_{ik}$:
  $$\pi_k = \frac{N_k}{n},\quad \mu_k = \frac{1}{N_k}\sum_i \gamma_{ik}x_i,\quad
    \sigma_k^2 = \frac{1}{N_k}\sum_i \gamma_{ik}(x_i-\mu_k)^2 .$$

*In words:* guess the classes, let every particle vote fractionally for each class according to how well it fits, update
each class from its votes, repeat. Each round can only **increase** (never decrease) the log-likelihood — a guarantee we
will check numerically.

In [ ]:
# ── 7.1 EM for a 1D Gaussian mixture, written by hand ──
rng = np.random.default_rng(SEED + 71)
size_feature = np.concatenate([rng.normal(11.0, 1.2, 200), rng.normal(15.0, 1.8, 100)])   # nm; truth: 2/3 vs 1/3
size_truth = np.repeat([0, 1], [200, 100])


def em_gmm_1d(x: np.ndarray, mu: np.ndarray, sd: np.ndarray, pi: np.ndarray, n_iter: int = 300):
    """EM for a 1D Gaussian mixture. Returns final (mu, sd, pi), responsibilities, log-likelihood trace, snapshots."""
    trace, snaps = [], {0: (mu.copy(), sd.copy(), pi.copy())}
    for it in range(1, n_iter + 1):
        dens = pi * stats.norm.pdf(x[:, None], mu, sd)          # n × K: π_k N(x_i | μ_k, σ_k²)
        trace.append(np.log(dens.sum(axis=1)).sum())            # log-likelihood of the current parameters
        gamma = dens / dens.sum(axis=1, keepdims=True)          # E-step
        Nk = gamma.sum(axis=0)                                  # M-step
        pi = Nk / len(x)
        mu = (gamma * x[:, None]).sum(axis=0) / Nk
        sd = np.sqrt((gamma * (x[:, None] - mu) ** 2).sum(axis=0) / Nk)
        if it in (1, 5):
            snaps[it] = (mu.copy(), sd.copy(), pi.copy())
    snaps[n_iter] = (mu, sd, pi)
    return (mu, sd, pi), gamma, np.array(trace), snaps


init = (np.array([9.0, 10.0]), np.array([3.0, 3.0]), np.array([0.5, 0.5]))     # a deliberately poor start
(mu1, sd1, pi1), gamma1, trace1, snaps1 = em_gmm_1d(size_feature, *init)
print("our EM     : μ =", np.round(mu1, 2), " σ =", np.round(sd1, 2), " π =", np.round(pi1, 2))
sk = GaussianMixture(2, tol=1e-8, max_iter=2000, random_state=SEED).fit(size_feature[:, None])   # tight tolerance
order = np.argsort(sk.means_.ravel())
print("scikit-learn: μ =", np.round(sk.means_.ravel()[order], 2), " σ =",
      np.round(np.sqrt(sk.covariances_.ravel()[order]), 2), " π =", np.round(sk.weights_[order], 2))
print("truth       : μ = [11. 15.]  σ = [1.2 1.8]  π = [0.67 0.33]")
print("log-likelihood never decreased:", bool(np.all(np.diff(trace1) > -1e-9)))

In [ ]:
# ── 7.2 Watch EM converge ──
grid = np.linspace(4, 22, 300)
fig, axes = plt.subplots(1, 5, figsize=(12, 2.6))
for ax, (it, (m, s, p)) in zip(axes[:4], snaps1.items()):
    ax.hist(size_feature, bins=40, density=True, color="lightgray")
    for k in range(2):
        ax.plot(grid, p[k] * stats.norm.pdf(grid, m[k], s[k]), lw=1.5)
    ax.plot(grid, sum(p[k] * stats.norm.pdf(grid, m[k], s[k]) for k in range(2)), "k--", lw=1)
    ax.set_title(f"iteration {it}", fontsize=9); ax.set_xlabel("feature (nm)"); ax.set_ylim(0, 0.3)
axes[0].set_ylabel("density")
axes[4].plot(np.arange(1, len(trace1) + 1), trace1, "-")
axes[4].set_xscale("log"); axes[4].set_title("log-likelihood ℓ", fontsize=9); axes[4].set_xlabel("iteration (log scale)")
plt.tight_layout(); plt.show()
print(f"particles with an uncertain label (0.2 < γ < 0.8): {np.sum((gamma1[:, 0] > 0.2) & (gamma1[:, 0] < 0.8))} of {len(size_feature)}")

🔍 **What just happened?** From a poor start (both components on the left), EM moved one Gaussian onto each population
within a few iterations; the log-likelihood rose monotonically and flattened. Fine-tuning in the overlap region is slow
(hundreds of iterations): scikit-learn's default stopping tolerance (`tol=1e-3`) would stop earlier, so we tightened it —
then our hand-written EM and scikit-learn agree.
The **responsibilities** are soft: particles in the overlap region keep split votes instead of being forced into one
class — the key idea that makes maximum-likelihood classification (FAML) robust at low SNR.

⚠️ EM finds a *local* optimum. Real implementations start from several initialisations and keep the best likelihood
(`n_init` in scikit-learn).

### 7.3 Two dimensions: features of real zoo subtomograms

Now real (simulated) cryo-ET data. We create **400 subtomograms** of our four molecules — unequal abundances, like a
cell (ribosomes most common) — each randomly rotated, at **SNR 0.05**. Each is summarised by its **radial profile**: the
mean density on concentric spherical shells around the centre. In Module 02 you saw that such shell features do not
change when the molecule rotates, so we need no alignment. Finally **principal component analysis (PCA)** compresses
the 14 shell values into 2 numbers: it finds the directions along which the particles vary most (PC1, PC2), like
viewing a cloud of points from its most informative angle.

In [ ]:
# ── 7.3 A library of noisy, randomly rotated subtomograms and their rotation-invariant features ──
rng = np.random.default_rng(SEED + 73)
def radial_profile(vol: np.ndarray, n_shells: int = 14) -> np.ndarray:
    """Mean density on concentric 1-voxel-thick spherical shells around the box centre (rotation-invariant)."""
    c = (np.array(vol.shape) - 1) / 2
    zz, yy, xx = np.indices(vol.shape)
    r = np.sqrt((zz - c[0]) ** 2 + (yy - c[1]) ** 2 + (xx - c[2]) ** 2)
    shell = np.minimum(r.astype(int), n_shells)
    sums = np.bincount(shell.ravel(), vol.ravel(), n_shells + 1)
    return (sums / np.bincount(shell.ravel(), minlength=n_shells + 1))[:n_shells]


t0 = time.time()
ABUNDANCE = {"4v4r": 150, "3j9i": 80, "6utj": 60, "1bxn": 110}       # copies per species in our toy "cell"
SNR_LIB = 0.05
labels_true, features, examples = [], [], {}
for k, (pid, count) in enumerate(ABUNDANCE.items()):
    for j in range(count):
        sub = add_noise_snr(rotate_volume(zoo[pid], random_rotation(rng)), SNR_LIB, rng)
        features.append(radial_profile(normalize(sub)))
        labels_true.append(k)
        if j == 0:
            examples[pid] = sub
labels_true, features = np.array(labels_true), np.array(features)
pca = PCA(n_components=2).fit(features)
Z = pca.transform(features)                                          # 400 × 2 feature matrix
print(f"{len(Z)} subtomograms in {time.time() - t0:.1f} s; PCA keeps {100 * pca.explained_variance_ratio_.sum():.0f} % of the variance")

fig, axes = plt.subplots(1, 5, figsize=(12, 2.7), gridspec_kw=dict(width_ratios=[1, 1, 1, 1, 1.6]))
for ax, (pid, sub) in zip(axes[:4], examples.items()):
    ax.imshow(sub[16]); ax.axis("off"); ax.set_title(f"{NAMES[pid]}\n(central slice)", fontsize=8)
for k, pid in enumerate(ABUNDANCE):
    axes[4].scatter(*Z[labels_true == k].T, s=5, label=NAMES[pid])
axes[4].set_xlabel("PC1"); axes[4].set_ylabel("PC2"); axes[4].legend(fontsize=6, markerscale=2)
axes[4].set_title("true species (unknown in practice)", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Single slices are hopeless to read by eye, yet in feature space the particles form groups:
ribosomes (large), RuBisCO (small), and the two proteasome variants (overlapping each other — the 15 nm cap barely changes a
spherical profile). Each group is an elongated cloud, so we need a mixture of Gaussians with **full covariance
matrices**. In two dimensions a Gaussian is

$$\mathcal N(\mathbf x\mid\boldsymbol\mu,\Sigma) = \frac{1}{2\pi\sqrt{\det\Sigma}}
  \exp\!\Big(-\tfrac12(\mathbf x-\boldsymbol\mu)^{\top}\Sigma^{-1}(\mathbf x-\boldsymbol\mu)\Big),$$

where the **covariance matrix** $\Sigma$ (2 × 2) encodes the spread along each axis and their tilt. The EM updates are the
same as in 1D, with $\Sigma_k = \frac{1}{N_k}\sum_i\gamma_{ik}(\mathbf x_i-\boldsymbol\mu_k)(\mathbf x_i-\boldsymbol\mu_k)^{\top}$.
We compute responsibilities in log space ("log-sum-exp") to avoid underflow.

In [ ]:
# ── 7.4 EM for a multivariate GMM with full covariances, by hand ──
def em_gmm(X: np.ndarray, K: int, n_iter: int = 200, seed: int = 0, tol: float = 1e-6):
    """Full-covariance Gaussian mixture fitted by EM. Returns dict(mu, cov, pi, gamma, loglik_trace)."""
    r = np.random.default_rng(seed)
    n, d = X.shape
    mu = X[r.choice(n, K, replace=False)]                  # start: K random particles as centres
    cov = np.array([np.cov(X.T) for _ in range(K)])
    pi = np.full(K, 1 / K)
    trace = []
    for _ in range(n_iter):
        logd = np.stack([np.log(pi[k]) + stats.multivariate_normal.logpdf(X, mu[k], cov[k]) for k in range(K)], 1)
        logp = np.logaddexp.reduce(logd, axis=1)            # log Σ_k π_k N(x_i | μ_k, Σ_k)
        trace.append(logp.sum())
        gamma = np.exp(logd - logp[:, None])                 # E-step
        Nk = gamma.sum(axis=0)                               # M-step
        pi = Nk / n
        mu = gamma.T @ X / Nk[:, None]
        for k in range(K):
            D = X - mu[k]
            cov[k] = (gamma[:, k, None] * D).T @ D / Nk[k] + 1e-6 * np.eye(d)
        if len(trace) > 1 and abs(trace[-1] - trace[-2]) < tol:
            break
    return dict(mu=mu, cov=cov, pi=pi, gamma=gamma, loglik_trace=np.array(trace))


fits = [em_gmm(Z, K=3, seed=s) for s in range(5)]            # 5 random starts, keep the best
ours = max(fits, key=lambda f: f["loglik_trace"][-1])
sk3 = GaussianMixture(3, covariance_type="full", n_init=5, random_state=SEED).fit(Z)
print("final log-likelihood of each of our 5 starts:", np.round([f["loglik_trace"][-1] for f in fits], 2))
print(f"best: ours {ours['loglik_trace'][-1]:.2f}  vs  scikit-learn {sk3.score(Z) * len(Z):.2f}")
print("our means:\n", np.round(ours["mu"][np.argsort(ours["mu"][:, 0])], 3))
print("sklearn means:\n", np.round(sk3.means_[np.argsort(sk3.means_[:, 0])], 3))


def draw_ellipse(ax, mu, cov, color, **kw):
    """Draw the 2-standard-deviation ellipse of a 2D Gaussian."""
    vals, vecs = np.linalg.eigh(cov)
    angle = np.degrees(np.arctan2(vecs[1, -1], vecs[0, -1]))
    ax.add_patch(Ellipse(mu, 4 * np.sqrt(vals[-1]), 4 * np.sqrt(vals[0]), angle=angle, fill=False, color=color, lw=2, **kw))


labels_ours = ours["gamma"].argmax(axis=1)
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharex=True, sharey=True)
for k in range(3):
    axes[0].scatter(*Z[labels_ours == k].T, s=5, color=f"C{k}")
    draw_ellipse(axes[0], ours["mu"][k], ours["cov"][k], f"C{k}")
    draw_ellipse(axes[1], sk3.means_[k], sk3.covariances_[k], "k", ls="--")
for k in range(4):
    axes[1].scatter(*Z[labels_true == k].T, s=5, alpha=0.6, label=NAMES[CAST[k]])
axes[0].set_title("our EM (K = 3): clusters (colours arbitrary) + 2σ ellipses"); axes[1].set_title("truth, with scikit-learn's ellipses")
axes[1].legend(fontsize=6, markerscale=2); axes[0].set_xlabel("PC1"); axes[1].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Our 25-line EM reaches the same log-likelihood and means as scikit-learn (different random
starts can end in different local optima — that is why we kept the best of five). With $K = 3$ it finds ribosomes,
RuBisCO and "proteasomes" (both variants merged). Is 3 the right number of clusters? Section 8 asks the data.

✏️ **Your turn.** Change `K=3` to `K=4` in `em_gmm(Z, K=3, ...)` and re-run. Does the fourth component separate the
capped from the uncapped proteasome, or does it cut a cloud in an arbitrary place?

### 📄 Paper connection — FAML: maximum likelihood with hidden classes *and* poses

**Paper.** Zhao Y, Zeng X, Guo Q, Xu M (2018). *An integration of fast alignment and maximum-likelihood methods for
electron subtomogram averaging and classification.* **Bioinformatics** 34(13):i227–i236 (ISMB 2018).
[doi:10.1093/bioinformatics/bty267](https://doi.org/10.1093/bioinformatics/bty267); arXiv:1804.01203.

- **Question it answers.** Maximum-likelihood (ML) averaging and classification is robust but must integrate over all
  rotations and translations (too slow); fast alignment is cheap but commits to one pose and fails at low SNR. Can we get both?
- **What the paper actually does.** A generative model $X_i = R_{\phi_i}A_{\kappa_i} + G_i$ with Gaussian noise $G_i$:
  each subtomogram is a rotated/shifted class average plus noise, using only Fourier coefficients inside the observed
  wedge. **EM** with hidden variables = class $k$ *and* pose $\phi$. The E-step computes posterior weights
  $P(k,\phi\mid X_i)\propto \alpha_k\exp\{-\lVert R_\phi A_k - X_i\rVert^2/2\sigma^2 - \dots\}$ over a **small set of
  candidate poses** (< 50) taken from the peaks of fast alignment, weighted by Voronoi volumes; the M-step updates the
  averages $A_k$, class priors $\alpha_k$, noise $\sigma^2$ and a translation prior.
- **What our toy keeps.** Hidden class labels, soft responsibilities, the E/M alternation and monotone likelihood — with
  class priors $\pi_k$ playing the role of $\alpha_k$.
- **What our toy leaves out.** Hidden *poses* (our features are rotation-invariant), missing wedges, class averages as
  3D maps, Voronoi weights. Module 14 builds a 2D FAML-like EM over poses.
- **Result reported.** At SNR 0.003 the authors report correlation with the true GroEL structure of 0.77 for FAML vs 0.19
  for fast alignment alone, and a mean F1 of 0.863 vs 0.462 on an experimental TRiC/proteasome mixture.
- **Code.** AITom `aitom/average/ml/faml/faml.py` (tutorial `doc/tutorials/009_faml.py`).

---
## 8 · How many clusters? The Bayesian information criterion (BIC)

More components always fit the data better (a mixture with one Gaussian per particle would be "perfect"), so the
likelihood alone cannot choose $K$. The **Bayesian information criterion** (Schwarz 1978) adds a penalty for the number
of free parameters $p$:

$$\mathrm{BIC}(K) = -2\,\ell_{\max}(K) + p(K)\,\ln n, \qquad
  p(K) = \underbrace{(K-1)}_{\text{weights}} + \underbrace{Kd}_{\text{means}} + \underbrace{K\,\tfrac{d(d+1)}{2}}_{\text{covariances}} .$$

*In words:* reward fit, charge $\ln n$ per parameter; choose the $K$ with the **lowest** BIC. The **AIC** is the same with
a charge of 2 per parameter (a weaker penalty, so it tends to pick larger $K$).

In [ ]:
# ── 8.1 Choose K by BIC (and AIC) ──
def n_params_gmm(K: int, d: int) -> int:
    """Number of free parameters of a full-covariance Gaussian mixture in d dimensions."""
    return (K - 1) + K * d + K * d * (d + 1) // 2


Ks = list(range(1, 9))
bic, aic = [], []
for K in Ks:
    gm = GaussianMixture(K, covariance_type="full", n_init=5, random_state=SEED).fit(Z)
    ll = gm.score(Z) * len(Z)                                   # total log-likelihood
    my_bic = -2 * ll + n_params_gmm(K, Z.shape[1]) * np.log(len(Z))
    assert np.isclose(my_bic, gm.bic(Z)), "our BIC formula should match scikit-learn"
    bic.append(my_bic); aic.append(gm.aic(Z))
K_bic = Ks[int(np.argmin(bic))]
print("BIC:", np.round(bic, 0)); print("AIC:", np.round(aic, 0))
print(f"BIC chooses K = {K_bic};  AIC chooses K = {Ks[int(np.argmin(aic))]};  true number of species = 4")
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(Ks, bic, "o-", label="BIC"); ax.plot(Ks, aic, "s--", label="AIC")
ax.axvline(K_bic, color="gray", ls=":"); ax.set_xlabel("number of components K"); ax.set_ylabel("criterion (lower is better)")
ax.set_title("model selection on the 2D zoo features"); ax.legend(fontsize=7); plt.tight_layout(); plt.show()

🔍 **What just happened?** Our parameter count reproduces scikit-learn's BIC exactly. BIC (and here AIC too) chooses
**K = 3, not 4**: the capped and uncapped proteasomes are not separable *in these features*, so the data do not support
a fourth cluster. **BIC counts the groups your features can see, not the species in the cell.** Better features — learned
by a neural network (DISCA) or computed from aligned voxels (Multi-Pattern Pursuit) — are what make finer distinctions
possible.

✏️ **Your turn.** (a) Raise `SNR_LIB` in cell 7.3 to 0.5 and re-run 7.3 → 8.1: does BIC now find 4? (b) Keep SNR 0.05
but use `PCA(n_components=4)`. (c) Remove `"6utj"` from `ABUNDANCE` — does BIC find 3?

### 📄 Paper connection — DISCA: GMM + BIC inside a deep-learning loop

**Paper.** Zeng X, Kahng A, Xue L, Mahamid J, Chang Y-W, Xu M (2023). *High-throughput cryo-ET structural pattern mining by
unsupervised deep iterative subtomogram clustering.* **PNAS** 120(15):e2213149120. [doi:10.1073/pnas.2213149120](https://doi.org/10.1073/pnas.2213149120)
(bioRxiv preprint titled *"DISCA: high-throughput cryo-ET structural pattern mining by deep unsupervised clustering"*).

- **Question it answers.** Sort hundreds of thousands of picked subtomograms into structurally homogeneous groups — and
  discover *how many* groups there are — with neither templates nor labels.
- **What the paper actually does.** A generalised EM loop: a CNN ("YOPO") extracts features; PCA reduces them to 16
  dimensions; a **full-covariance GMM** is fitted for several candidate $K$ and the **K with the lowest BIC** is kept;
  labels are matched to the previous iteration with the **Hungarian algorithm** (Section 10); the CNN is retrained on the
  new (smoothed) labels with rotation augmentation; the best iteration is chosen with a distortion-based Davies–Bouldin index.
- **What our toy keeps.** PCA features → full-covariance GMM → BIC over K.
- **What our toy leaves out.** The learned CNN features (ours are fixed radial profiles), the iterative retraining and
  the Davies–Bouldin model selection — all in Module 22.
- **Result reported.** On *Mycoplasma pneumoniae* (198,715 subtomograms from 65 tomograms) the authors report a 14.17 Å
  ribosome average and ribosome F1 0.893 vs 0.766 for template matching; on simulated data 99.70 % accuracy vs 83.95 % for
  template matching at SNR 0.1.
- **Code.** AITom `aitom/classify/deep/unsupervised/disca/` (Keras `DISCA.py`, PyTorch `torch_disca.py`).

---
## 9 · k-means, hierarchical clustering and the silhouette

### 9.1 k-means: hard assignments, round clusters

**k-means** is the simplest clustering method: choose $K$ centres, assign every point to its nearest centre, move each
centre to the mean of its points, repeat (Lloyd's algorithm). It minimises the **inertia**

$$J = \sum_{i}\lVert \mathbf x_i - \boldsymbol\mu_{c(i)}\rVert^2 ,$$

*in words:* the total squared distance of points to their cluster centre. k-means is "EM with hard labels and identical
round Gaussians". It is fast and is used everywhere — Multi-Pattern Pursuit generates candidate patterns with it, and
the fungal-membrane study checked its clusters with k-means inertia. Because inertia always falls as $K$ grows, people
look for an **"elbow"** in the inertia curve.

In [ ]:
# ── 9.1 k-means by hand (Lloyd's algorithm) vs scikit-learn, and the elbow ──
def kmeans_lloyd(X: np.ndarray, K: int, n_iter: int = 100, seed: int = 0):
    """Plain k-means: returns (labels, centres, inertia)."""
    r = np.random.default_rng(seed)
    centres = X[r.choice(len(X), K, replace=False)]
    for _ in range(n_iter):
        d2 = ((X[:, None, :] - centres[None]) ** 2).sum(-1)     # squared distance to every centre
        lab = d2.argmin(axis=1)                                  # assignment step
        new = np.array([X[lab == k].mean(0) if np.any(lab == k) else centres[k] for k in range(K)])
        if np.allclose(new, centres):
            break
        centres = new                                            # update step
    return lab, centres, d2[np.arange(len(X)), lab].sum()


best = min((kmeans_lloyd(Z, 3, seed=s) for s in range(10)), key=lambda t: t[2])
km3 = KMeans(3, n_init=10, random_state=SEED).fit(Z)
print(f"inertia — ours: {best[2]:.3f}   scikit-learn: {km3.inertia_:.3f}")
inertias = [KMeans(K, n_init=10, random_state=SEED).fit(Z).inertia_ for K in Ks]
fig, ax = plt.subplots(figsize=(4.6, 2.8))
ax.plot(Ks, inertias, "o-"); ax.set_xlabel("K"); ax.set_ylabel("inertia J"); ax.set_title("k-means elbow curve")
plt.tight_layout(); plt.show()

### 9.2 Hierarchical clustering cut by the silhouette (the Xu, Beck & Alber 2012 recipe)

**Hierarchical (agglomerative) clustering** starts with every particle as its own cluster and repeatedly merges the two
closest clusters, producing a tree (**dendrogram**). With **average linkage**, the distance between two clusters is the
average of all pairwise distances between their members. We need a pairwise distance between particles. Following Xu,
Beck & Alber (2012), we turn a correlation $c_{jk}$ into a distance

$$d_{jk} = \sqrt{2 - 2c_{jk}},$$

which is exactly the Euclidean distance between two standardised vectors (zero mean, unit length). In that paper $c_{jk}$
is the constrained correlation between aligned subtomograms (Module 12); here we use the correlation between rotation-
invariant radial profiles. Where to cut the tree? The **silhouette** of particle $i$ compares its mean distance to its
own cluster, $a(i)$, with its mean distance to the nearest other cluster, $b(i)$:

$$s(i) = \frac{b(i)-a(i)}{\max\{a(i),\,b(i)\}} \in [-1, 1].$$

*In words:* $s\approx1$ — snugly inside its cluster; $s\approx0$ — on a border; $s<0$ — probably in the wrong cluster.
We cut the tree at the $K$ with the largest **average silhouette width**.

In [ ]:
# ── 9.2 Average-linkage clustering on d = sqrt(2 − 2c) and the silhouette ──
C = np.corrcoef(features)                                  # 400 × 400 Pearson correlations of radial profiles
D = np.sqrt(np.clip(2 - 2 * C, 0, None)); np.fill_diagonal(D, 0)
tree = hierarchy.linkage(squareform(D, checks=False), method="average")
sil = {}
for K in range(2, 9):
    lab = hierarchy.fcluster(tree, K, criterion="maxclust") - 1
    sil[K] = silhouette_score(D, lab, metric="precomputed")
K_sil = max(sil, key=sil.get)
labels_hier = hierarchy.fcluster(tree, K_sil, criterion="maxclust") - 1
print("average silhouette width:", {k: round(v, 3) for k, v in sil.items()}, "→ cut at K =", K_sil)
print("cluster sizes at that cut:", np.bincount(labels_hier))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw=dict(width_ratios=[2, 1]))
hierarchy.dendrogram(tree, ax=a1, no_labels=True, color_threshold=tree[-K_sil + 1, 2], above_threshold_color="gray")
a1.set_ylabel("average-linkage distance"); a1.set_title(f"dendrogram of 400 subtomograms (coloured at K = {K_sil})")
a2.plot(list(sil), list(sil.values()), "o-"); a2.set_xlabel("K (tree cut)"); a2.set_ylabel("average silhouette")
a2.set_title("choose the cut with the best silhouette"); plt.tight_layout(); plt.show()

🔍 **What just happened?** The first split of the tree separates one tight, very distinct group (orange, 110
particles) from all the rest: RuBisCO, as the contingency table in Section 10.1 confirms. The silhouette — which rewards
tight, well-separated groups — is highest at **K = 4**, the true number of species, yet the four clusters are *not* the
four species: one holds both proteasome variants (140 = 80 + 60), and the ribosomes are split into a large group (135)
and a small one (15). A silhouette maximum marks the *most obvious* splits, not the number of species — always inspect
the clusters before trusting a criterion. GMM + BIC found three groups (the k-means elbow is sharpest at K = 2, with a
smaller bend at 3); when the features cannot separate two species (the proteasome variants), no criterion will find
them. Next we score all these clusterings against the truth.

### 📄 Paper connection — alignment-through-classification (Xu, Beck & Alber 2012)

**Paper.** Xu M, Beck M, Alber F (2012). *High-throughput subtomogram alignment and classification by Fourier space
constrained fast volumetric matching.* **J Struct Biol** 178(2):152–164. [doi:10.1016/j.jsb.2012.02.014](https://doi.org/10.1016/j.jsb.2012.02.014)

- **Question it answers.** How to align and classify thousands of subtomograms reference-free, despite missing wedges.
- **What the paper actually does (classification part).** Pairwise wedge-aware **constrained correlations** $c_{jk}$
  (with feature-enhancement weights from voxel covariances) → distances $d_{jk}=\sqrt{2-2c_{jk}}$ → **average-linkage
  hierarchical clustering**, tree cut at the **maximal average silhouette width** → class averages → re-align every
  subtomogram to every average and reassign → iterate until the silhouette plateaus.
- **What our toy keeps.** The distance transform, average linkage and the silhouette cut.
- **What our toy leaves out.** Alignment and the constrained correlation (Module 12), feature-enhancement weights and the
  iteration (Module 14).
- **Result reported.** 100 % classification of four simulated complexes (with feature enhancement), 68.2 % on 786
  experimental GroEL/GroEL–GroES subtomograms, and 99 % for ribosome vs ATP synthase in *Leptospira interrogans* tomograms.
- **Code.** AITom `aitom/align/fast/` (alignment); the classification scheme is described in the paper.

---
## 10 · Scoring results: Hungarian accuracy, ARI, precision, recall, F1

### 10.1 Cluster labels are arbitrary — match them first

A clustering algorithm might call the ribosomes "cluster 2". To compute an accuracy we must first find the best
one-to-one matching between clusters and true classes. Count the **contingency table** $n_{kc}$ (particles of class $c$
in cluster $k$) and choose the matching that maximises the matched counts. Trying all matchings would take $K!$ steps;
the **Hungarian algorithm** (Kuhn 1955) solves this *assignment problem* efficiently (`scipy.optimize.linear_sum_assignment`):

$$\text{accuracy} = \frac{1}{n}\max_{\text{one-to-one } m}\ \sum_{k} n_{k,\,m(k)} .$$

Clusters left unmatched (when $K$ ≠ number of classes) count as errors. DISCA uses the same Hungarian matching to keep
cluster labels consistent between iterations.

In [ ]:
# ── 10.1 Contingency tables and Hungarian-matched accuracy ──
def contingency(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    """n[k, c] = number of particles of true class c placed in cluster k."""
    ct = np.zeros((y_pred.max() + 1, y_true.max() + 1), int)
    np.add.at(ct, (y_pred, y_true), 1)
    return ct


clusterings = {
    "GMM, K = 3 (BIC)": GaussianMixture(3, covariance_type="full", n_init=5, random_state=SEED).fit(Z).predict(Z),
    "GMM, K = 4": GaussianMixture(4, covariance_type="full", n_init=5, random_state=SEED).fit(Z).predict(Z),
    "k-means, K = 3": km3.labels_,
    f"hierarchical, K = {K_sil}": labels_hier,
    "GMM, K = 6 (over-split)": GaussianMixture(6, covariance_type="full", n_init=5, random_state=SEED).fit(Z).predict(Z),
}
fig, axes = plt.subplots(1, len(clusterings), figsize=(13, 2.9))
for ax, (name, lab) in zip(axes, clusterings.items()):
    ct = contingency(labels_true, lab)
    ax.imshow(ct, cmap="viridis")
    for (k, c), v in np.ndenumerate(ct):
        ax.text(c, k, v, ha="center", va="center", fontsize=7, color="w" if v < ct.max() / 2 else "k")
    ax.set_xticks(range(4)); ax.set_xticklabels([p.upper() for p in CAST], fontsize=6, rotation=45)
    ax.set_xlabel("true class", fontsize=7); ax.set_ylabel("cluster", fontsize=7)
    ax.set_title(f"{name}\nacc = {hungarian_accuracy(labels_true, lab):.2f}", fontsize=8)
plt.tight_layout(); plt.show()

🔍 **What just happened?** In each table a good clustering has one dominant cell per row and column. With $K=3$ the two
proteasome columns land in the same cluster, so the best possible accuracy is 1 − 60/400 = 0.85 (one of the two proteasome
classes must be "wrong"). With $K=6$ the extra clusters cannot be matched to any class and count as errors.

### 10.2 The adjusted Rand index (ARI): agreement on *pairs*

Accuracy requires a matching; the **Rand index** avoids it by looking at all $\binom n2$ **pairs** of particles: a pair
is handled correctly if both partitions put it together, or both put it apart. The **adjusted** Rand index (Hubert &
Arabie 1985) subtracts the agreement expected by chance, so a random clustering scores ≈ 0 and a perfect one exactly 1.
From the contingency table $n_{kc}$ with row sums $a_k$ and column sums $b_c$:

$$\mathrm{ARI}=\frac{\sum_{kc}\binom{n_{kc}}{2}-E}{\tfrac12\Big[\sum_k\binom{a_k}{2}+\sum_c\binom{b_c}{2}\Big]-E},\qquad
  E=\frac{\sum_k\binom{a_k}{2}\sum_c\binom{b_c}{2}}{\binom n2}.$$

*In words:* count pairs that are together in both partitions, compare with what chance would give, and rescale.

In [ ]:
# ── 10.2 ARI by hand vs scikit-learn ──
def comb2(x):
    """Number of pairs x·(x−1)/2 (works elementwise on arrays)."""
    return x * (x - 1) / 2


def adjusted_rand_index(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    """Adjusted Rand index from the contingency table (Hubert & Arabie 1985)."""
    ct = contingency(y_true, y_pred)
    sum_cells = comb2(ct).sum()
    sum_a, sum_b = comb2(ct.sum(axis=1)).sum(), comb2(ct.sum(axis=0)).sum()
    expected = sum_a * sum_b / comb2(len(y_true))
    return float((sum_cells - expected) / (0.5 * (sum_a + sum_b) - expected))


print(f"{'clustering':26s} {'Hungarian acc':>13s} {'ARI (ours)':>10s} {'ARI (sklearn)':>13s}")
for name, lab in clusterings.items():
    print(f"{name:26s} {hungarian_accuracy(labels_true, lab):13.3f} {adjusted_rand_index(labels_true, lab):10.3f} "
          f"{adjusted_rand_score(labels_true, lab):13.3f}")
perm = np.array([2, 0, 3, 1])[labels_true]                 # truth with renamed labels
rand_lab = rng.integers(0, 4, len(labels_true))            # random guessing
print(f"{'truth, labels renamed':26s} {hungarian_accuracy(labels_true, perm):13.3f} {adjusted_rand_index(labels_true, perm):10.3f}")
print(f"{'random labels':26s} {hungarian_accuracy(labels_true, rand_lab):13.3f} {adjusted_rand_index(labels_true, rand_lab):10.3f}")

🔍 **What just happened?** Our ARI matches scikit-learn. Renaming labels does not matter (accuracy and ARI both 1).
Random labels give ARI ≈ 0 but an accuracy well above zero (≈ 0.3 here) — raw accuracy has a "chance floor" that depends
on the class sizes, which is why papers report ARI (or normalised mutual information) next to accuracy.

### 10.3 Detection: precision, recall and F1

Picking asks a yes/no question per candidate: *is this a ribosome?* Comparing predictions with the truth gives true
positives (TP), false positives (FP) and false negatives (FN):

$$\text{precision}=\frac{TP}{TP+FP},\qquad \text{recall}=\frac{TP}{TP+FN},\qquad
  F_1=\frac{2\,\text{precision}\cdot\text{recall}}{\text{precision}+\text{recall}} .$$

*In words:* precision — of what I picked, how much is right; recall — of what exists, how much I found; F1 — their
harmonic mean, high only when both are. A score threshold trades one against the other. Below, each candidate is scored
by how well its radial profile correlates with the noise-free ribosome's profile (a rotation-invariant "template match").

In [ ]:
# ── 10.3 Precision–recall curve of a simple ribosome detector ──
ribo_profile = radial_profile(normalize(zoo["4v4r"]))
scores = np.array([pearson(f, ribo_profile) for f in features])
is_ribo = labels_true == 0
thresholds = np.sort(scores)[::-1]
P, R, F = [], [], []
for t in thresholds:
    pred = scores >= t
    tp, fp, fn = np.sum(pred & is_ribo), np.sum(pred & ~is_ribo), np.sum(~pred & is_ribo)
    P.append(tp / (tp + fp)); R.append(tp / (tp + fn)); F.append(2 * tp / (2 * tp + fp + fn))
i_best = int(np.argmax(F))
print(f"best F1 = {F[i_best]:.3f} at threshold {thresholds[i_best]:.3f} (precision {P[i_best]:.3f}, recall {R[i_best]:.3f})")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3))
a1.hist(scores[is_ribo], bins=30, alpha=0.6, label="ribosomes"); a1.hist(scores[~is_ribo], bins=30, alpha=0.6, label="other molecules")
a1.axvline(thresholds[i_best], color="k", ls="--", label="best-F1 threshold"); a1.set_xlabel("score (correlation of profiles)")
a1.set_ylabel("count"); a1.legend(fontsize=7)
a2.plot(R, P, "-"); a2.scatter(R[i_best], P[i_best], color="tab:red", zorder=3, label=f"best F1 = {F[i_best]:.2f}")
a2.set_xlabel("recall"); a2.set_ylabel("precision"); a2.set_title("precision–recall curve"); a2.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Ribosomes score higher than other molecules, but the two score distributions overlap a
little, so no threshold is perfect: lowering it raises recall and lowers precision. F1 summarises the best compromise.
You will meet these numbers in nearly every paper: DISCA's ribosome F1 of 0.893 vs 0.766 for template matching, the SHREC
picking benchmarks (Modules 05, 16), TomoPicker and CryoMAE (Module 31). For picking in 3D, a prediction is "true" if it
falls close enough to a real particle centre (one-to-one matching, toolkit `match_points`, Module 11).

---
## ✅ Checkpoint

1. A subtomogram has SNR 0.02. How many perfectly aligned copies must you average to reach SNR 2?
   <details><summary>Answer</summary>SNR grows like N, so N = 2 / 0.02 = 100 copies (noise std shrinks 10-fold).</details>
2. Two independent noisy copies of the same particle correlate with c = 0.09. What is the SNR?
   <details><summary>Answer</summary>SNR = c/(1 − c) = 0.09/0.91 ≈ 0.099 ≈ 0.1.</details>
3. Why does the Wilcoxon signed-rank test fit MPP's redundancy question better than comparing two independent groups?
   <details><summary>Answer</summary>Each particle is scored against both patterns, so the data are paired; analysing the per-particle differences removes the large particle-to-particle variation (noise, orientation) that both scores share.</details>
4. You test 2,000 hexamers from 2 membranes per condition and get p = 10⁻⁴⁰. What is wrong?
   <details><summary>Answer</summary>Pseudo-replication: hexamers in one membrane are not independent samples. The unit of replication is the membrane (or biological replicate); with 2 per group no test can give a convincing p-value.</details>
5. In one sentence each: E-step and M-step of EM for a GMM.
   <details><summary>Answer</summary>E-step: compute each particle's probability (responsibility) of belonging to each component under the current parameters. M-step: re-estimate each component's weight, mean and covariance as responsibility-weighted averages.</details>
6. BIC chose K = 3 for four species. Name two ways to change that.
   <details><summary>Answer</summary>Use more informative features (e.g., learned CNN features as in DISCA, or aligned voxels as in MPP), or improve the data (higher SNR, more particles). BIC can only find the groups the features separate.</details>

## 🧠 Questions & discussion

1. Averaging improves SNR $N$-fold — so why not average a million subtomograms of a cell regardless of species?
   What is lost, and what does that teach about the order "classify, then average"?
2. In the membrane example, the effect was large but p could not drop below 0.002 with 6 vs 6 patches. How would you
   design an experiment (how many cells, membranes, tomograms) to test a *small* drug effect convincingly?
3. MPP drops pattern A if its members do not significantly prefer it at α = 0.01. What happens to a genuinely distinct
   but *rare* complex with only 10 members? (Recall: MPP often missed complexes with < 200–300 copies.)
4. GMMs give soft responsibilities; k-means gives hard labels. Which would you trust at very low SNR, and why does FAML
   keep *several* candidate poses per particle instead of the single best one?
5. Accuracy, ARI, F1: which would you report for (a) a picker, (b) an unsupervised clustering with unknown K, (c) a
   supervised classifier with very unequal class sizes?

## 🚀 Bonus — going further

- **SNR in Fourier space.** Split the 1,024 copies of Section 4 into two halves, average each, and compute the
  correlation of the two half-averages *per frequency ring* — you have just invented the Fourier ring correlation (Module 10).
- **Bootstrap.** Resample the patch medians with replacement 10,000 times and build a 95 % confidence interval for the
  control-vs-drug difference. Compare with the permutation test.
- **Mixtures of other shapes.** Fit `GaussianMixture(covariance_type="spherical")` and `"diag"` to `Z`; compare their
  BIC with `"full"`. Which assumption fits elongated clouds?
- **Deep dives in this repository.** [`../../../../ml/tutorial/FundamentalsLearningUncertainty/`](../../../../ml/tutorial/FundamentalsLearningUncertainty/)
  (probability, likelihood, uncertainty) and [`../../../../ml/tutorial/GenerativeDiscriminativeClassification/`](../../../../ml/tutorial/GenerativeDiscriminativeClassification/)
  (generative vs discriminative classifiers).
- **Lab code.** Read `aitom/average/ml/faml/faml.py` and find the E-step (`ln_p_k_phi`) and M-step (`update_*`) functions.

## 📝 Lab note

```text
Module 04 — date:
Input: zoo maps (4V4R, 3J9I, 6UTJ, 1BXN) → noisy rotated subtomograms (SNR 0.05); toy membrane point patterns
Output: SNR estimates, averages vs N, p-values (permutation, Kruskal–Wallis, Wilcoxon), GMM/k-means/hierarchical clusterings
Method: c/(1−c) SNR; √N averaging; rank tests; EM for GMM; BIC; silhouette; Hungarian accuracy; ARI; F1
Parameter I changed and what happened:
A failure mode / surprise (e.g., BIC merging the proteasome variants; pseudo-replication):
Paper link: FAML (Zhao 2018); MPP (Xu 2019); DISCA (Zeng 2023); fungal membrane (Jiang 2025); Xu, Beck & Alber 2012
```

## 📚 References

1. Frank J, Al-Ali L. Signal-to-noise ratio of electron micrographs obtained by cross correlation. *Nature* 256:376–379 (1975). doi:10.1038/256376a0
2. Dempster AP, Laird NM, Rubin DB. Maximum likelihood from incomplete data via the EM algorithm. *J R Stat Soc B* 39:1–38 (1977). doi:10.1111/j.2517-6161.1977.tb01600.x
3. Schwarz G. Estimating the dimension of a model. *Ann Stat* 6:461–464 (1978). doi:10.1214/aos/1176344136
4. Wilcoxon F. Individual comparisons by ranking methods. *Biometrics Bulletin* 1:80–83 (1945). doi:10.2307/3001968
5. Kruskal WH, Wallis WA. Use of ranks in one-criterion variance analysis. *J Am Stat Assoc* 47:583–621 (1952). doi:10.1080/01621459.1952.10483441
6. Dunn OJ. Multiple comparisons using rank sums. *Technometrics* 6:241–252 (1964). doi:10.1080/00401706.1964.10490181
7. Kuhn HW. The Hungarian method for the assignment problem. *Naval Res Logist Q* 2:83–97 (1955). doi:10.1002/nav.3800020109
8. Hubert L, Arabie P. Comparing partitions. *J Classif* 2:193–218 (1985). doi:10.1007/BF01908075
9. Rousseeuw PJ. Silhouettes: a graphical aid to the interpretation and validation of cluster analysis. *J Comput Appl Math* 20:53–65 (1987). doi:10.1016/0377-0427(87)90125-7
10. Xu M, Beck M, Alber F. High-throughput subtomogram alignment and classification by Fourier space constrained fast volumetric matching. *J Struct Biol* 178:152–164 (2012). doi:10.1016/j.jsb.2012.02.014
11. Zhao Y, Zeng X, Guo Q, Xu M. An integration of fast alignment and maximum-likelihood methods for electron subtomogram averaging and classification. *Bioinformatics* 34:i227–i236 (2018). doi:10.1093/bioinformatics/bty267
12. Xu M, Singla J, Tocheva EI, Chang Y-W, Stevens RC, Jensen GJ, Alber F. De novo structural pattern mining in cellular electron cryotomograms. *Structure* 27:679–691.e14 (2019). doi:10.1016/j.str.2019.01.005
13. Zeng X, Kahng A, Xue L, Mahamid J, Chang Y-W, Xu M. High-throughput cryo-ET structural pattern mining by unsupervised deep iterative subtomogram clustering. *PNAS* 120:e2213149120 (2023). doi:10.1073/pnas.2213149120
14. Jiang J, …, Xu M, Perlin DS, Dai W. Molecular landscape of the fungal plasma membrane and implications for antifungal action. *Nat Commun* 16:9125 (2025). doi:10.1038/s41467-025-64171-x
15. Jiang R, Daggett J, Pingulkar S, Zhao Y, Dhingra P, Brown D, Wu Q, Zeng X, Li X, Xu M. BOE-ViT: Boosting orientation estimation with equivariance in self-supervised 3D subtomogram alignment. *CVPR* 2025, pp. 29352–29362.

## ➡️ Next

**[05 · Deep learning in one afternoon (PyTorch)](05_deep_learning_pytorch.ipynb)** — you will train neural networks
that classify noisy molecules, and see how data leakage and rotation augmentation make or break them. Course overview:
[`README.md`](../README.md) · terms: [`GLOSSARY.md`](../GLOSSARY.md).